<a href="https://colab.research.google.com/github/Mrsayem/Secure-LoRA-Adapter/blob/main/secure_atomic_lora_Final_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%pip -q install "transformers>=4.45,<6" "peft>=0.14,<1" "accelerate>=0.30" "safetensors>=0.4" "cryptography>=42" "pandas>=2" "psutil>=5"


In [ ]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
%%writefile secure_atomic_lora_colab.py

from __future__ import annotations

import contextvars
import copy
import hashlib
import json
import math
import os
import random
import shutil
import statistics
import threading
import time
import tempfile
import uuid
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Mapping, MutableMapping, Optional, Sequence, Tuple
from types import MappingProxyType

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ed25519
from safetensors import safe_open
from safetensors.torch import load_file, save_file

SEED = 20260722
SCHEMA_VERSION = 1
GENESIS_HASH = "0" * 64
PACKAGE_FILENAME = "adapter.safetensors"
MANIFEST_FILENAME = "manifest.json"
SIGNATURE_FILENAME = "signature.hex"
DEFAULT_CHUNK_BYTES = 1 << 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

class SecurityError(RuntimeError):
    """Raised when a package violates the admission policy."""


class CrashInjected(RuntimeError):
    """Used only by the crash-recovery experiment."""


def canonical_json_bytes(value: Any) -> bytes:
    """Deterministic JSON encoding for signatures and hashes."""
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: os.PathLike[str] | str, block_size: int = 4 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def dtype_name(tensor: torch.Tensor) -> str:
    return str(tensor.dtype).replace("torch.", "")


def tensor_raw_bytes(tensor: torch.Tensor) -> bytes:
    """Return exact storage bytes and support bfloat16 without NumPy casting."""
    cpu = tensor.detach().to("cpu").contiguous()
    return cpu.view(torch.uint8).numpy().tobytes()




def tensor_chunk_raw_bytes(tensor: torch.Tensor, offset: int, length: int) -> bytes:
    """Copy only one byte range from a tensor to host memory.

    This avoids the original audit implementation's hidden O(tensor_size) host
    transfer for every sampled chunk. The tensor is first viewed as bytes on its
    current device, sliced there, and only the requested range is copied to CPU.
    """
    if offset < 0 or length < 0:
        raise ValueError("offset and length must be non-negative")
    byte_view = tensor.detach().contiguous().view(torch.uint8).reshape(-1)
    if offset > byte_view.numel():
        raise ValueError("offset exceeds tensor byte length")
    chunk = byte_view[offset : offset + length].to("cpu", non_blocking=False).contiguous()
    return chunk.numpy().tobytes()


def dtype_nbytes(dtype: str) -> int:
    sizes = {
        "float64": 8,
        "float32": 4,
        "float16": 2,
        "bfloat16": 2,
        "int64": 8,
        "int32": 4,
        "int16": 2,
        "int8": 1,
        "uint8": 1,
        "bool": 1,
    }
    try:
        return sizes[dtype]
    except KeyError as exc:
        raise SecurityError(f"unsupported tensor dtype in manifest: {dtype!r}") from exc


def tensor_record(name: str, tensor: torch.Tensor) -> Dict[str, Any]:
    raw = tensor_raw_bytes(tensor)
    metadata = {
        "name": name,
        "dtype": dtype_name(tensor),
        "shape": list(tensor.shape),
        "nbytes": len(raw),
    }
    digest = sha256_bytes(canonical_json_bytes(metadata) + b"\0" + raw)
    return {**metadata, "sha256": digest}


def verify_tensor_record(name: str, tensor: torch.Tensor, expected: Mapping[str, Any]) -> None:
    actual = tensor_record(name, tensor)
    for field_name in ("name", "dtype", "shape", "nbytes", "sha256"):
        if actual[field_name] != expected.get(field_name):
            raise SecurityError(
                f"tensor {name!r} failed {field_name} check: "
                f"expected={expected.get(field_name)!r}, actual={actual[field_name]!r}"
            )


def hash_model_state(model: nn.Module) -> str:
    """Hash names, dtypes, shapes, lengths, and bytes of the full state_dict."""
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        record = tensor_record(name, tensor)
        digest.update(canonical_json_bytes(record))
        digest.update(b"\n")
    return digest.hexdigest()


def synchronize(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def percentile(values: Sequence[float], q: float) -> float:
    return float(np.percentile(np.asarray(values, dtype=np.float64), q))


def bootstrap_ratio_ci(
    numerator: Sequence[float],
    denominator: Sequence[float],
    n_bootstrap: int = 2000,
    seed: int = SEED,
) -> Tuple[float, float]:
    """95% bootstrap CI for median(numerator)/median(denominator) - 1."""
    rng = np.random.default_rng(seed)
    a = np.asarray(numerator, dtype=np.float64)
    b = np.asarray(denominator, dtype=np.float64)
    samples = []
    for _ in range(n_bootstrap):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        samples.append(np.median(aa) / np.median(bb) - 1.0)
    return percentile(samples, 2.5), percentile(samples, 97.5)


class UpdateAuthority:
    """Trusted update pipeline. The private key must stay off the serving host."""

    def __init__(self, private_key: Optional[ed25519.Ed25519PrivateKey] = None):
        self._private_key = private_key or ed25519.Ed25519PrivateKey.generate()
        self.public_key = self._private_key.public_key()
        public_raw = self.public_key.public_bytes(
            encoding=serialization.Encoding.Raw,
            format=serialization.PublicFormat.Raw,
        )
        self.key_id = sha256_bytes(public_raw)

    def sign(self, manifest: Mapping[str, Any]) -> bytes:
        return self._private_key.sign(canonical_json_bytes(manifest))

    def verify(self, manifest: Mapping[str, Any], signature: bytes) -> None:
        self.public_key.verify(signature, canonical_json_bytes(manifest))


@dataclass(frozen=True)
class WitnessHead:
    deployment_id: str
    version: int
    manifest_hash: str


class OnlineWitness:
    def __init__(self, public_key: ed25519.Ed25519PublicKey, key_id: str):
        self.public_key = public_key
        self.key_id = key_id
        self._heads: Dict[str, WitnessHead] = {}
        self._lock = threading.RLock()

    def read(self, deployment_id: str) -> WitnessHead:
        with self._lock:
            return self._heads.get(
                deployment_id,
                WitnessHead(deployment_id=deployment_id, version=0, manifest_hash=GENESIS_HASH),
            )

    def commit(
        self,
        manifest: Mapping[str, Any],
        signature: bytes,
        manifest_hash: str,
    ) -> WitnessHead:
        try:
            self.public_key.verify(signature, canonical_json_bytes(manifest))
        except InvalidSignature as exc:
            raise SecurityError("witness rejected invalid authority signature") from exc

        deployment_id = str(manifest["deployment_id"])
        with self._lock:
            current = self.read(deployment_id)
            version = int(manifest["adapter_version"])
            parent = str(manifest["parent_manifest_hash"])
            if manifest.get("signing_key_id") != self.key_id:
                raise SecurityError("witness rejected unknown signing key")
            if version != current.version + 1:
                raise SecurityError(
                    f"witness expected version {current.version + 1}, received {version}"
                )
            if parent != current.manifest_hash:
                raise SecurityError("witness parent hash does not match current head")
            head = WitnessHead(deployment_id, version, manifest_hash)
            self._heads[deployment_id] = head
            return head
@dataclass(frozen=True)
class TargetSpec:
    module_name: str
    in_features: int
    out_features: int

    @property
    def a_key(self) -> str:
        return f"{self.module_name}.lora_A"

    @property
    def b_key(self) -> str:
        return f"{self.module_name}.lora_B"


@dataclass(frozen=True)
class AdapterRuntime:
    adapter_id: str
    version: int
    manifest_hash: str
    rank: int
    alpha: float
    tensors: Mapping[str, torch.Tensor]
    manifest: Mapping[str, Any]
    expected_chunk_hashes: Mapping[str, Tuple[str, ...]] = field(default_factory=dict)

    @property
    def scale(self) -> float:
        return self.alpha / self.rank


@dataclass(frozen=True)
class PinnedRequest:
    adapter_id: Optional[str]
    version: int
    manifest_hash: str
    observed_manifest_hashes: set[str]


_CURRENT_ADAPTER: contextvars.ContextVar[Optional[AdapterRuntime]] = contextvars.ContextVar(
    "current_adapter", default=None
)
_CURRENT_TRACE: contextvars.ContextVar[Optional[set[str]]] = contextvars.ContextVar(
    "current_adapter_trace", default=None
)


class VersionedLoRAModule(nn.Module):
    def __init__(self, base: nn.Module, spec: TargetSpec):
        super().__init__()
        self.base = base
        self.spec = spec
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)

    def forward(self, x: torch.Tensor, *args: Any, **kwargs: Any) -> torch.Tensor:
        base_output = self.base(x, *args, **kwargs)
        adapter = _CURRENT_ADAPTER.get()
        if adapter is None:
            return base_output

        trace = _CURRENT_TRACE.get()
        if trace is not None:
            trace.add(adapter.manifest_hash)

        a = adapter.tensors.get(self.spec.a_key)
        b = adapter.tensors.get(self.spec.b_key)
        if a is None or b is None:
            return base_output

        lora_input = x.to(dtype=a.dtype)
        update = F.linear(F.linear(lora_input, a), b) * adapter.scale
        return base_output + update.to(dtype=base_output.dtype)


def _get_parent_and_leaf(model: nn.Module, qualified_name: str) -> Tuple[nn.Module, str]:
    parts = qualified_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


def discover_lora_targets(
    model: nn.Module,
    target_suffixes: Sequence[str],
    max_modules: Optional[int] = None,
) -> List[TargetSpec]:
    specs: List[TargetSpec] = []
    for name, module in model.named_modules():
        if not name or not any(name.endswith(suffix) for suffix in target_suffixes):
            continue
        weight = getattr(module, "weight", None)
        if not isinstance(weight, torch.Tensor) or weight.ndim != 2:
            continue

        if isinstance(module, nn.Linear):
            in_features = int(module.in_features)
            out_features = int(module.out_features)
        elif module.__class__.__name__ == "Conv1D":
            # Transformers GPT-2 Conv1D stores [in_features, out_features].
            in_features = int(weight.shape[0])
            out_features = int(weight.shape[1])
        else:
            continue

        specs.append(TargetSpec(name, in_features, out_features))
        if max_modules is not None and len(specs) >= max_modules:
            break

    if not specs:
        raise ValueError(
            "No compatible target modules found. Inspect model.named_modules() and "
            "change TARGET_SUFFIXES."
        )
    return specs


def inject_versioned_lora(model: nn.Module, specs: Sequence[TargetSpec]) -> None:
    module_map = dict(model.named_modules())
    for spec in specs:
        original = module_map[spec.module_name]
        parent, leaf = _get_parent_and_leaf(model, spec.module_name)
        setattr(parent, leaf, VersionedLoRAModule(original, spec))
    for parameter in model.parameters():
        parameter.requires_grad_(False)


# -----------------------------------------------------------------------------
# Package creation
# -----------------------------------------------------------------------------

def build_random_lora_tensors(
    specs: Sequence[TargetSpec],
    rank: int,
    seed: int,
    dtype: torch.dtype = torch.float32,
    std: float = 0.02,
) -> Dict[str, torch.Tensor]:
    generator = torch.Generator(device="cpu").manual_seed(seed)
    tensors: Dict[str, torch.Tensor] = {}
    for spec in specs:
        a = torch.randn(rank, spec.in_features, generator=generator, dtype=dtype) * std
        b = torch.randn(spec.out_features, rank, generator=generator, dtype=dtype) * std
        tensors[spec.a_key] = a.contiguous()
        tensors[spec.b_key] = b.contiguous()
    return tensors


def expected_adapter_schema(specs: Sequence[TargetSpec], rank: int) -> Dict[str, Tuple[List[int], str]]:
    schema: Dict[str, Tuple[List[int], str]] = {}
    for spec in specs:
        schema[spec.a_key] = ([rank, spec.in_features], "A")
        schema[spec.b_key] = ([spec.out_features, rank], "B")
    return schema


def write_signed_package(
    output_dir: os.PathLike[str] | str,
    authority: UpdateAuthority,
    witness: OnlineWitness,
    deployment_id: str,
    base_model_id: str,
    base_model_hash: str,
    adapter_id: str,
    adapter_version: int,
    rank: int,
    alpha: float,
    tensors: Mapping[str, torch.Tensor],
    target_specs: Sequence[TargetSpec],
    ttl_seconds: int = 3600,
    created_at: Optional[int] = None,
    manifest_overrides: Optional[Mapping[str, Any]] = None,
) -> Path:
    """Create adapter.safetensors, manifest.json, and signature.hex."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=False)
    created = int(time.time()) if created_at is None else int(created_at)

    cpu_tensors = {name: tensor.detach().cpu().contiguous() for name, tensor in tensors.items()}
    package_path = output / PACKAGE_FILENAME
    save_file(
        cpu_tensors,
        str(package_path),
        metadata={
            "format": "secure-atomic-lora",
            "adapter_id": adapter_id,
            "adapter_version": str(adapter_version),
        },
    )

    current = witness.read(deployment_id)
    manifest: Dict[str, Any] = {
        "schema_version": SCHEMA_VERSION,
        "deployment_id": deployment_id,
        "base_model_id": base_model_id,
        "base_model_hash": base_model_hash,
        "adapter_id": adapter_id,
        "adapter_version": int(adapter_version),
        "parent_manifest_hash": current.manifest_hash,
        "created_at": created,
        "expires_at": created + int(ttl_seconds),
        "signing_key_id": authority.key_id,
        "rank": int(rank),
        "alpha": float(alpha),
        "target_modules": [spec.module_name for spec in target_specs],
        "tensors": {name: tensor_record(name, tensor) for name, tensor in sorted(cpu_tensors.items())},
        "package_sha256": sha256_file(package_path),
    }
    if manifest_overrides:
        manifest.update(copy.deepcopy(dict(manifest_overrides)))

    signature = authority.sign(manifest)
    (output / MANIFEST_FILENAME).write_bytes(canonical_json_bytes(manifest))
    (output / SIGNATURE_FILENAME).write_text(signature.hex(), encoding="ascii")
    return output


def load_package_files(package_dir: os.PathLike[str] | str) -> Tuple[Dict[str, Any], bytes, Path]:
    package = Path(package_dir)
    try:
        manifest = json.loads((package / MANIFEST_FILENAME).read_text(encoding="utf-8"))
        signature = bytes.fromhex((package / SIGNATURE_FILENAME).read_text(encoding="ascii").strip())
    except (OSError, ValueError, json.JSONDecodeError) as exc:
        raise SecurityError(f"malformed package metadata: {exc}") from exc
    return manifest, signature, package / PACKAGE_FILENAME

class SecureAdapterManager:
    def __init__(
        self,
        model: nn.Module,
        target_specs: Sequence[TargetSpec],
        deployment_id: str,
        base_model_id: str,
        base_model_hash: str,
        authority_public_key: ed25519.Ed25519PublicKey,
        signing_key_id: str,
        witness: OnlineWitness,
        device: torch.device,
        chunk_bytes: int = DEFAULT_CHUNK_BYTES,
        clock_skew_seconds: int = 5,
        allowed_adapter_dtypes: Sequence[str] = ("float32",),
        max_rank: int = 256,
        max_adapter_bytes: int = 2 << 30,
        max_package_bytes: int = 3 << 30,
    ):
        self.model = model
        self.target_specs = list(target_specs)
        self.deployment_id = deployment_id
        self.base_model_id = base_model_id
        self.base_model_hash = base_model_hash
        self.public_key = authority_public_key
        self.signing_key_id = signing_key_id
        self.witness = witness
        self.device = device
        self.chunk_bytes = int(chunk_bytes)
        self.clock_skew_seconds = int(clock_skew_seconds)
        self.allowed_adapter_dtypes = frozenset(str(x) for x in allowed_adapter_dtypes)
        self.max_rank = int(max_rank)
        self.max_adapter_bytes = int(max_adapter_bytes)
        self.max_package_bytes = int(max_package_bytes)
        if self.chunk_bytes <= 0:
            raise ValueError("chunk_bytes must be positive")
        if min(self.max_rank, self.max_adapter_bytes, self.max_package_bytes) <= 0:
            raise ValueError("resource limits must be positive")
        self._active: Optional[AdapterRuntime] = None
        self._lock = threading.RLock()
        self._package_catalog: Dict[str, Path] = {}
        self._last_audit_bytes_read = 0

    @property
    def active_version(self) -> int:
        with self._lock:
            return 0 if self._active is None else self._active.version

    @property
    def active_manifest_hash(self) -> str:
        with self._lock:
            return GENESIS_HASH if self._active is None else self._active.manifest_hash

    def _verify_manifest_signature(self, manifest: Mapping[str, Any], signature: bytes) -> None:
        try:
            self.public_key.verify(signature, canonical_json_bytes(manifest))
        except InvalidSignature as exc:
            raise SecurityError("invalid manifest signature") from exc

    def _validate_manifest_common(self, manifest: Mapping[str, Any], now: Optional[int] = None) -> None:
        required = {
            "schema_version", "deployment_id", "base_model_id", "base_model_hash",
            "adapter_id", "adapter_version", "parent_manifest_hash", "created_at",
            "expires_at", "signing_key_id", "rank", "alpha", "target_modules",
            "tensors", "package_sha256",
        }
        missing = required.difference(manifest)
        if missing:
            raise SecurityError(f"manifest missing fields: {sorted(missing)}")
        if int(manifest["schema_version"]) != SCHEMA_VERSION:
            raise SecurityError("unsupported manifest schema version")
        if manifest["deployment_id"] != self.deployment_id:
            raise SecurityError("package belongs to a different deployment")
        if manifest["base_model_id"] != self.base_model_id:
            raise SecurityError("package names a different base model")
        if manifest["base_model_hash"] != self.base_model_hash:
            raise SecurityError("package base-model hash mismatch")
        if manifest["signing_key_id"] != self.signing_key_id:
            raise SecurityError("unknown signing key ID")
        if not isinstance(manifest["adapter_id"], str) or not manifest["adapter_id"]:
            raise SecurityError("adapter_id must be a non-empty string")
        if int(manifest["adapter_version"]) <= 0:
            raise SecurityError("adapter version must be positive")
        if int(manifest["rank"]) <= 0:
            raise SecurityError("rank must be positive")
        if int(manifest["rank"]) > self.max_rank:
            raise SecurityError(
                f"rank {manifest['rank']} exceeds deployment maximum {self.max_rank}"
            )
        if not math.isfinite(float(manifest["alpha"])) or float(manifest["alpha"]) <= 0:
            raise SecurityError("alpha must be finite and positive")

        current_time = int(time.time()) if now is None else int(now)
        created_at = int(manifest["created_at"])
        expires_at = int(manifest["expires_at"])
        if created_at > current_time + self.clock_skew_seconds:
            raise SecurityError("manifest creation time is in the future")
        if expires_at < current_time - self.clock_skew_seconds:
            raise SecurityError("manifest has expired")
        if expires_at <= created_at:
            raise SecurityError("manifest expiry must be after creation")

        expected_targets = [spec.module_name for spec in self.target_specs]
        if list(manifest["target_modules"]) != expected_targets:
            raise SecurityError("target module list does not match deployment policy")

    def _validate_freshness(self, manifest: Mapping[str, Any]) -> None:
        head = self.witness.read(self.deployment_id)
        expected_version = head.version + 1
        if int(manifest["adapter_version"]) != expected_version:
            raise SecurityError(
                f"expected next adapter version {expected_version}, "
                f"received {manifest['adapter_version']}"
            )
        if manifest["parent_manifest_hash"] != head.manifest_hash:
            raise SecurityError("parent manifest hash does not match witness head")

    def _validate_tensor_schema(self, manifest: Mapping[str, Any]) -> None:
        rank = int(manifest["rank"])
        expected = expected_adapter_schema(self.target_specs, rank)
        declared = manifest["tensors"]
        if not isinstance(declared, dict):
            raise SecurityError("manifest tensors must be an object")
        if set(declared) != set(expected):
            extra = sorted(set(declared) - set(expected))
            missing = sorted(set(expected) - set(declared))
            raise SecurityError(f"adapter tensor scope mismatch; extra={extra}, missing={missing}")
        total_declared_bytes = 0
        for name, (shape, _) in expected.items():
            record = declared[name]
            if not isinstance(record, Mapping):
                raise SecurityError(f"tensor record for {name!r} must be an object")
            if record.get("name") != name:
                raise SecurityError(f"tensor record for {name!r} has a different embedded name")
            if list(record.get("shape", [])) != shape:
                raise SecurityError(
                    f"tensor {name!r} shape violates LoRA policy: "
                    f"expected={shape}, declared={record.get('shape')}"
                )
            declared_dtype = str(record.get("dtype", ""))
            if declared_dtype not in self.allowed_adapter_dtypes:
                raise SecurityError(
                    f"tensor {name!r} dtype {declared_dtype!r} violates deployment policy; "
                    f"allowed={sorted(self.allowed_adapter_dtypes)}"
                )
            try:
                declared_nbytes = int(record.get("nbytes", -1))
            except (TypeError, ValueError) as exc:
                raise SecurityError(f"tensor {name!r} has a non-integer byte length") from exc
            expected_nbytes = math.prod(shape) * dtype_nbytes(declared_dtype)
            if declared_nbytes != expected_nbytes:
                raise SecurityError(
                    f"tensor {name!r} byte length violates schema: "
                    f"expected={expected_nbytes}, declared={declared_nbytes}"
                )
            total_declared_bytes += declared_nbytes
        if total_declared_bytes > self.max_adapter_bytes:
            raise SecurityError(
                f"adapter payload {total_declared_bytes} bytes exceeds deployment maximum "
                f"{self.max_adapter_bytes}"
            )

    def _build_chunk_hashes(self, tensors: Mapping[str, torch.Tensor]) -> Dict[str, Tuple[str, ...]]:
        output: Dict[str, Tuple[str, ...]] = {}
        for name, tensor in sorted(tensors.items()):
            total_bytes = tensor.numel() * tensor.element_size()
            output[name] = tuple(
                sha256_bytes(
                    tensor_chunk_raw_bytes(
                        tensor, offset, min(self.chunk_bytes, total_bytes - offset)
                    )
                )
                for offset in range(0, total_bytes, self.chunk_bytes)
            )
        return output

    def _copy_to_private_staging(self, source: Path, destination: Path) -> None:
        """Copy an untrusted package file once, then verify and load that private copy.

        The original code hashed a path and reopened the same path later, leaving a
        hash-to-load TOCTOU window. A final hash over this private copy binds the
        exact bytes subsequently parsed by safetensors.
        """
        flags = os.O_RDONLY | getattr(os, "O_NOFOLLOW", 0)
        try:
            fd = os.open(source, flags)
        except OSError as exc:
            raise SecurityError(f"cannot open adapter package safely: {exc}") from exc
        try:
            with os.fdopen(fd, "rb", closefd=True) as src, open(destination, "xb") as dst:
                copied = 0
                while True:
                    block = src.read(4 << 20)
                    if not block:
                        break
                    copied += len(block)
                    if copied > self.max_package_bytes:
                        raise SecurityError(
                            f"adapter package exceeds deployment maximum {self.max_package_bytes} bytes"
                        )
                    dst.write(block)
        except Exception:
            destination.unlink(missing_ok=True)
            raise

    def verify_and_stage(
        self,
        package_dir: os.PathLike[str] | str,
        *,
        require_next_version: bool = True,
        now: Optional[int] = None,
    ) -> Tuple[AdapterRuntime, Dict[str, Any], bytes]:
        manifest, signature, tensor_path = load_package_files(package_dir)
        self._verify_manifest_signature(manifest, signature)
        self._validate_manifest_common(manifest, now=now)
        if require_next_version:
            self._validate_freshness(manifest)
        self._validate_tensor_schema(manifest)

        if not tensor_path.is_file():
            raise SecurityError("adapter.safetensors is missing")

        declared = manifest["tensors"]
        try:
            with tempfile.TemporaryDirectory(prefix="secure-lora-stage-") as staging_dir:
                staged_path = Path(staging_dir) / PACKAGE_FILENAME
                self._copy_to_private_staging(tensor_path, staged_path)
                if sha256_file(staged_path) != manifest["package_sha256"]:
                    raise SecurityError("Safetensors package hash mismatch")
                with safe_open(str(staged_path), framework="pt", device="cpu") as handle:
                    actual_keys = set(handle.keys())
                    if actual_keys != set(declared):
                        raise SecurityError(
                            f"Safetensors key set mismatch; extra={sorted(actual_keys-set(declared))}, "
                            f"missing={sorted(set(declared)-actual_keys)}"
                        )
                    cpu_tensors = {
                        name: handle.get_tensor(name).contiguous() for name in sorted(actual_keys)
                    }
        except SecurityError:
            raise
        except Exception as exc:
            raise SecurityError(f"invalid Safetensors package: {exc}") from exc

        for name, tensor in cpu_tensors.items():
            verify_tensor_record(name, tensor, declared[name])

        manifest_hash = sha256_bytes(canonical_json_bytes(manifest))
        device_tensors = {
            name: tensor.to(self.device, non_blocking=False).contiguous()
            for name, tensor in cpu_tensors.items()
        }
        runtime = AdapterRuntime(
            adapter_id=str(manifest["adapter_id"]),
            version=int(manifest["adapter_version"]),
            manifest_hash=manifest_hash,
            rank=int(manifest["rank"]),
            alpha=float(manifest["alpha"]),
            tensors=MappingProxyType(device_tensors),
            manifest=MappingProxyType(copy.deepcopy(dict(manifest))),
            expected_chunk_hashes=MappingProxyType(self._build_chunk_hashes(device_tensors)),
        )
        return runtime, dict(manifest), signature

    def admit(
        self,
        package_dir: os.PathLike[str] | str,
        warmup: Optional[Callable[[AdapterRuntime], None]] = None,
        crash_point: Optional[str] = None,
    ) -> AdapterRuntime:
        runtime, manifest, signature = self.verify_and_stage(package_dir, require_next_version=True)
        if warmup is not None:
            warmup(runtime)

        # Register the fully verified package before the witness commit. If the
        # process crashes after commit, recovery can locate this exact package.
        self._package_catalog[runtime.manifest_hash] = Path(package_dir)
        if crash_point == "before_witness_commit":
            raise CrashInjected("crash before witness commit")

        self.witness.commit(manifest, signature, runtime.manifest_hash)
        if crash_point == "after_witness_commit":
            raise CrashInjected("crash after witness commit")

        with self._lock:
            self._active = runtime  # Atomic object-reference swap under the lock.
        return runtime

    def recover_from_witness(self, package_catalog: Mapping[str, os.PathLike[str] | str]) -> Optional[AdapterRuntime]:
        head = self.witness.read(self.deployment_id)
        if head.version == 0:
            with self._lock:
                self._active = None
            return None
        package_dir = package_catalog.get(head.manifest_hash)
        if package_dir is None:
            raise SecurityError("witness head exists but its verified package is unavailable")
        runtime, _, _ = self.verify_and_stage(package_dir, require_next_version=False)
        if runtime.version != head.version or runtime.manifest_hash != head.manifest_hash:
            raise SecurityError("recovery package does not match witness head")
        with self._lock:
            self._active = runtime
        self._package_catalog.update({str(k): Path(v) for k, v in package_catalog.items()})
        return runtime

    @contextmanager
    def pin_active(self, trace: bool = True) -> Iterable[PinnedRequest]:
        with self._lock:
            runtime = self._active
        observed: set[str] = set()
        adapter_token = _CURRENT_ADAPTER.set(runtime)
        trace_token = _CURRENT_TRACE.set(observed if trace else None)
        try:
            yield PinnedRequest(
                adapter_id=None if runtime is None else runtime.adapter_id,
                version=0 if runtime is None else runtime.version,
                manifest_hash=GENESIS_HASH if runtime is None else runtime.manifest_hash,
                observed_manifest_hashes=observed,
            )
        finally:
            _CURRENT_TRACE.reset(trace_token)
            _CURRENT_ADAPTER.reset(adapter_token)

    @contextmanager
    def use_runtime(self, runtime: Optional[AdapterRuntime], trace: bool = False) -> Iterable[None]:
        observed: set[str] = set()
        adapter_token = _CURRENT_ADAPTER.set(runtime)
        trace_token = _CURRENT_TRACE.set(observed if trace else None)
        try:
            yield
        finally:
            _CURRENT_TRACE.reset(trace_token)
            _CURRENT_ADAPTER.reset(adapter_token)

    @property
    def last_audit_bytes_read(self) -> int:
        return int(self._last_audit_bytes_read)

    def audit_round(self, q_chunks: int, rng: Optional[random.Random] = None) -> Tuple[bool, List[str]]:
        rng = rng or random.SystemRandom()
        with self._lock:
            runtime = self._active
        if runtime is None:
            self._last_audit_bytes_read = 0
            return True, []

        population: List[Tuple[str, int]] = []
        for name, hashes in runtime.expected_chunk_hashes.items():
            population.extend((name, index) for index in range(len(hashes)))
        if not population:
            self._last_audit_bytes_read = 0
            return True, []

        q = min(max(1, int(q_chunks)), len(population))
        sampled = rng.sample(population, q)
        failures: List[str] = []
        bytes_read = 0
        for name, chunk_index in sampled:
            tensor = runtime.tensors[name]
            total_bytes = tensor.numel() * tensor.element_size()
            start = chunk_index * self.chunk_bytes
            length = min(self.chunk_bytes, total_bytes - start)
            raw = tensor_chunk_raw_bytes(tensor, start, length)
            bytes_read += len(raw)
            actual = sha256_bytes(raw)
            expected = runtime.expected_chunk_hashes[name][chunk_index]
            if actual != expected:
                failures.append(f"{name}#chunk{chunk_index}")
        self._last_audit_bytes_read = bytes_read
        return len(failures) == 0, failures

    def corrupt_active_tensor(self, tensor_name: str, flat_index: int = 0, delta: float = 1.0) -> None:
        """Attack injection used only by experiments."""
        with self._lock:
            runtime = self._active
        if runtime is None:
            raise RuntimeError("no active adapter")
        if tensor_name not in runtime.tensors:
            raise KeyError(tensor_name)
        with torch.no_grad():
            flat = runtime.tensors[tensor_name].view(-1)
            flat[flat_index] += delta

def make_warmup(
    manager: SecureAdapterManager,
    model_inputs: Mapping[str, torch.Tensor],
) -> Callable[[AdapterRuntime], None]:
    def warmup(runtime: AdapterRuntime) -> None:
        with torch.inference_mode(), manager.use_runtime(runtime):
            manager.model(**model_inputs)
        synchronize(manager.device)
    return warmup


def run_pinned_forward(
    manager: SecureAdapterManager,
    model_inputs: Mapping[str, torch.Tensor],
    repeats: int = 1,
    delay_seconds: float = 0.0,
) -> Dict[str, Any]:
    with torch.inference_mode(), manager.pin_active(trace=True) as pinned:
        for _ in range(repeats):
            manager.model(**model_inputs)
            if delay_seconds:
                time.sleep(delay_seconds)
        synchronize(manager.device)
        return {
            "pinned_version": pinned.version,
            "pinned_manifest_hash": pinned.manifest_hash,
            "observed_hashes": sorted(pinned.observed_manifest_hashes),
            "single_version": len(pinned.observed_manifest_hashes) <= 1,
        }


def clone_package(source: os.PathLike[str] | str, destination: os.PathLike[str] | str) -> Path:
    return Path(shutil.copytree(source, destination))


def resign_manifest(package_dir: os.PathLike[str] | str, authority: UpdateAuthority) -> None:
    package = Path(package_dir)
    manifest = json.loads((package / MANIFEST_FILENAME).read_text(encoding="utf-8"))
    (package / MANIFEST_FILENAME).write_bytes(canonical_json_bytes(manifest))
    (package / SIGNATURE_FILENAME).write_text(authority.sign(manifest).hex(), encoding="ascii")


def update_manifest(
    package_dir: os.PathLike[str] | str,
    authority: UpdateAuthority,
    updates: Mapping[str, Any],
) -> None:
    package = Path(package_dir)
    manifest = json.loads((package / MANIFEST_FILENAME).read_text(encoding="utf-8"))
    manifest.update(copy.deepcopy(dict(updates)))
    (package / MANIFEST_FILENAME).write_bytes(canonical_json_bytes(manifest))
    (package / SIGNATURE_FILENAME).write_text(authority.sign(manifest).hex(), encoding="ascii")


def replace_package_tensors_and_resign(
    package_dir: os.PathLike[str] | str,
    authority: UpdateAuthority,
    tensors: Mapping[str, torch.Tensor],
) -> None:
    package = Path(package_dir)
    tensor_path = package / PACKAGE_FILENAME
    save_file({k: v.detach().cpu().contiguous() for k, v in tensors.items()}, str(tensor_path))
    manifest = json.loads((package / MANIFEST_FILENAME).read_text(encoding="utf-8"))
    manifest["tensors"] = {name: tensor_record(name, tensor) for name, tensor in sorted(tensors.items())}
    manifest["package_sha256"] = sha256_file(tensor_path)
    (package / MANIFEST_FILENAME).write_bytes(canonical_json_bytes(manifest))
    (package / SIGNATURE_FILENAME).write_text(authority.sign(manifest).hex(), encoding="ascii")


def run_attack_suite(
    root_dir: os.PathLike[str] | str,
    model: nn.Module,
    target_specs: Sequence[TargetSpec],
    base_model_id: str,
    base_model_hash: str,
    device: torch.device,
    model_inputs: Mapping[str, torch.Tensor],
    rank: int = 4,
    alpha: float = 8.0,
) -> pd.DataFrame:
    """Run independent attacks with fresh witness/manager state per test."""
    root = Path(root_dir)
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True)
    rows: List[Dict[str, Any]] = []

    def fresh(test_name: str) -> Tuple[UpdateAuthority, OnlineWitness, SecureAdapterManager, Path, Dict[str, torch.Tensor]]:
        test_dir = root / test_name
        test_dir.mkdir()
        authority = UpdateAuthority()
        witness = OnlineWitness(authority.public_key, authority.key_id)
        deployment = f"deployment-{test_name}"
        manager = SecureAdapterManager(
            model=model,
            target_specs=target_specs,
            deployment_id=deployment,
            base_model_id=base_model_id,
            base_model_hash=base_model_hash,
            authority_public_key=authority.public_key,
            signing_key_id=authority.key_id,
            witness=witness,
            device=device,
            chunk_bytes=256,
        )
        tensors = build_random_lora_tensors(target_specs, rank=rank, seed=SEED + 1)
        package = write_signed_package(
            test_dir / "valid_v1",
            authority, witness, deployment, base_model_id, base_model_hash,
            adapter_id="adapter-A", adapter_version=1, rank=rank, alpha=alpha,
            tensors=tensors, target_specs=target_specs,
        )
        return authority, witness, manager, package, tensors

    def record(name: str, expected_reject: bool, action: Callable[[], Any]) -> None:
        started = time.perf_counter()
        rejected = False
        detail = "accepted"
        try:
            action()
        except (SecurityError, InvalidSignature, ValueError, RuntimeError) as exc:
            rejected = True
            detail = f"{type(exc).__name__}: {exc}"
        rows.append({
            "attack": name,
            "expected_reject": expected_reject,
            "rejected": rejected,
            "pass": rejected == expected_reject,
            "detail": detail,
            "elapsed_ms": (time.perf_counter() - started) * 1000.0,
        })

    # Valid package
    authority, witness, manager, package, tensors = fresh("valid_package")
    record("valid_package", False, lambda: manager.admit(package, make_warmup(manager, model_inputs)))

    # Forged signature
    authority, witness, manager, package, tensors = fresh("forged_signature")
    forged = clone_package(package, root / "forged_signature" / "attack")
    (forged / SIGNATURE_FILENAME).write_text(os.urandom(64).hex(), encoding="ascii")
    record("forged_signature", True, lambda: manager.admit(forged))

    # Tampered bytes after signing
    authority, witness, manager, package, tensors = fresh("tampered_bytes")
    tampered = clone_package(package, root / "tampered_bytes" / "attack")
    raw = bytearray((tampered / PACKAGE_FILENAME).read_bytes())
    raw[-1] ^= 0x01
    (tampered / PACKAGE_FILENAME).write_bytes(raw)
    record("tampered_bytes", True, lambda: manager.admit(tampered))

    # Signed but out-of-policy extra tensor
    authority, witness, manager, package, tensors = fresh("extra_tensor")
    extra = clone_package(package, root / "extra_tensor" / "attack")
    bad = dict(tensors)
    bad["unauthorized.extra"] = torch.ones(1)
    replace_package_tensors_and_resign(extra, authority, bad)
    record("extra_tensor", True, lambda: manager.admit(extra))

    # Signed but missing tensor
    authority, witness, manager, package, tensors = fresh("missing_tensor")
    missing = clone_package(package, root / "missing_tensor" / "attack")
    bad = dict(tensors)
    bad.pop(next(iter(bad)))
    replace_package_tensors_and_resign(missing, authority, bad)
    record("missing_tensor", True, lambda: manager.admit(missing))

    # Signed dtype substitution
    authority, witness, manager, package, tensors = fresh("dtype_substitution")
    dtype_attack = clone_package(package, root / "dtype_substitution" / "attack")
    bad = dict(tensors)
    key = next(iter(bad))
    bad[key] = bad[key].to(torch.float16)
    replace_package_tensors_and_resign(dtype_attack, authority, bad)
    record("dtype_substitution", True, lambda: manager.admit(dtype_attack))

    # Signed shape substitution
    authority, witness, manager, package, tensors = fresh("shape_substitution")
    shape_attack = clone_package(package, root / "shape_substitution" / "attack")
    bad = dict(tensors)
    key = next(iter(bad))
    bad[key] = bad[key].reshape(-1)
    replace_package_tensors_and_resign(shape_attack, authority, bad)
    record("shape_substitution", True, lambda: manager.admit(shape_attack))

    # Mix-and-match one tensor after signing.
    authority, witness, manager, package, tensors = fresh("mix_and_match")
    mixed = clone_package(package, root / "mix_and_match" / "attack")
    second = build_random_lora_tensors(target_specs, rank=rank, seed=SEED + 99)
    mixed_tensors = load_file(str(mixed / PACKAGE_FILENAME), device="cpu")
    mixed_key = next(iter(mixed_tensors))
    mixed_tensors[mixed_key] = second[mixed_key]
    # Intentionally do not update the signed manifest.
    save_file(mixed_tensors, str(mixed / PACKAGE_FILENAME))
    record("mix_and_match_after_signing", True, lambda: manager.admit(mixed))

    # Signed record whose embedded tensor name disagrees with its dictionary key.
    authority, witness, manager, package, tensors = fresh("embedded_name_mismatch")
    embedded = clone_package(package, root / "embedded_name_mismatch" / "attack")
    embedded_manifest = json.loads((embedded / MANIFEST_FILENAME).read_text(encoding="utf-8"))
    embedded_key = next(iter(embedded_manifest["tensors"]))
    embedded_manifest["tensors"][embedded_key]["name"] = "different.tensor.name"
    (embedded / MANIFEST_FILENAME).write_bytes(canonical_json_bytes(embedded_manifest))
    (embedded / SIGNATURE_FILENAME).write_text(authority.sign(embedded_manifest).hex(), encoding="ascii")
    record("embedded_tensor_name_mismatch", True, lambda: manager.admit(embedded))

    # Wrong deployment, correctly signed
    authority, witness, manager, package, tensors = fresh("wrong_deployment")
    wrong_deployment = clone_package(package, root / "wrong_deployment" / "attack")
    update_manifest(wrong_deployment, authority, {"deployment_id": "some-other-deployment"})
    record("wrong_deployment", True, lambda: manager.admit(wrong_deployment))

    # Wrong base model, correctly signed
    authority, witness, manager, package, tensors = fresh("wrong_base_model")
    wrong_base = clone_package(package, root / "wrong_base_model" / "attack")
    update_manifest(wrong_base, authority, {"base_model_hash": "f" * 64})
    record("wrong_base_model", True, lambda: manager.admit(wrong_base))

    # Wrong signing key identifier even though the same authority signed the bytes.
    authority, witness, manager, package, tensors = fresh("wrong_key_id")
    wrong_key = clone_package(package, root / "wrong_key_id" / "attack")
    update_manifest(wrong_key, authority, {"signing_key_id": "0" * 64})
    record("wrong_signing_key_id", True, lambda: manager.admit(wrong_key))

    # Version gap / fast-forward attempt.
    authority, witness, manager, package, tensors = fresh("version_skip")
    skipped = clone_package(package, root / "version_skip" / "attack")
    update_manifest(skipped, authority, {"adapter_version": 2})
    record("version_skip", True, lambda: manager.admit(skipped))

    # Creation time too far in the future.
    authority, witness, manager, package, tensors = fresh("future_manifest")
    future = clone_package(package, root / "future_manifest" / "attack")
    future_time = int(time.time()) + 3600
    update_manifest(future, authority, {"created_at": future_time, "expires_at": future_time + 3600})
    record("future_manifest", True, lambda: manager.admit(future))

    # Invalid LoRA scaling policy.
    authority, witness, manager, package, tensors = fresh("negative_alpha")
    negative_alpha = clone_package(package, root / "negative_alpha" / "attack")
    update_manifest(negative_alpha, authority, {"alpha": -1.0})
    record("negative_alpha", True, lambda: manager.admit(negative_alpha))

    # Expired package, correctly signed
    authority, witness, manager, package, tensors = fresh("expired_manifest")
    expired = clone_package(package, root / "expired_manifest" / "attack")
    old = int(time.time()) - 1000
    update_manifest(expired, authority, {"created_at": old - 100, "expires_at": old})
    record("expired_manifest", True, lambda: manager.admit(expired))

    # Empty signed package
    authority, witness, manager, package, tensors = fresh("empty_package")
    empty = clone_package(package, root / "empty_package" / "attack")
    replace_package_tensors_and_resign(empty, authority, {})
    record("empty_package", True, lambda: manager.admit(empty))

    # Signed resource-exhaustion attempt with an out-of-policy rank.
    authority, witness, manager, package, tensors = fresh("oversized_rank")
    oversized_rank = clone_package(package, root / "oversized_rank" / "attack")
    update_manifest(oversized_rank, authority, {"rank": manager.max_rank + 1})
    record("oversized_rank", True, lambda: manager.admit(oversized_rank))

    # Signed inconsistent byte count should be rejected before tensor allocation.
    authority, witness, manager, package, tensors = fresh("invalid_declared_nbytes")
    invalid_nbytes = clone_package(package, root / "invalid_declared_nbytes" / "attack")
    invalid_manifest = json.loads((invalid_nbytes / MANIFEST_FILENAME).read_text(encoding="utf-8"))
    invalid_key = next(iter(invalid_manifest["tensors"]))
    invalid_manifest["tensors"][invalid_key]["nbytes"] += 1
    (invalid_nbytes / MANIFEST_FILENAME).write_bytes(canonical_json_bytes(invalid_manifest))
    (invalid_nbytes / SIGNATURE_FILENAME).write_text(
        authority.sign(invalid_manifest).hex(), encoding="ascii"
    )
    record("invalid_declared_nbytes", True, lambda: manager.admit(invalid_nbytes))

    # A symlinked payload is rejected; admission must parse only a private staged copy.
    authority, witness, manager, package, tensors = fresh("symlinked_payload")
    symlinked = clone_package(package, root / "symlinked_payload" / "attack")
    payload = symlinked / PACKAGE_FILENAME
    target = symlinked / "payload-target.safetensors"
    payload.rename(target)
    payload.symlink_to(target.name)
    record("symlinked_payload", True, lambda: manager.admit(symlinked))

    # Replay after successful admission
    authority, witness, manager, package, tensors = fresh("replay")
    manager.admit(package)
    record("replay_stale_version", True, lambda: manager.admit(package))

    # Crash before witness commit must not advance freshness.
    authority, witness, manager, package, tensors = fresh("crash_before_commit")
    def crash_before() -> None:
        try:
            manager.admit(package, crash_point="before_witness_commit")
        except CrashInjected:
            pass
        if witness.read(manager.deployment_id).version != 0:
            raise RuntimeError("witness advanced before commit")
    record("crash_before_commit", False, crash_before)

    # Crash after witness commit, then recover exact package.
    authority, witness, manager, package, tensors = fresh("crash_after_commit")
    def crash_after_and_recover() -> None:
        catalog: Dict[str, Path] = {}
        try:
            manager.admit(package, crash_point="after_witness_commit")
        except CrashInjected:
            manifest = json.loads((package / MANIFEST_FILENAME).read_text(encoding="utf-8"))
            manifest_hash = sha256_bytes(canonical_json_bytes(manifest))
            catalog[manifest_hash] = package
        restarted = SecureAdapterManager(
            model=model,
            target_specs=target_specs,
            deployment_id=manager.deployment_id,
            base_model_id=base_model_id,
            base_model_hash=base_model_hash,
            authority_public_key=authority.public_key,
            signing_key_id=authority.key_id,
            witness=witness,
            device=device,
            chunk_bytes=256,
        )
        restarted.recover_from_witness(catalog)
        if restarted.active_version != 1:
            raise RuntimeError("recovery did not activate witness head")
    record("crash_after_commit_recovery", False, crash_after_and_recover)

    # Post-activation corruption should be found by complete chunk sweep.
    authority, witness, manager, package, tensors = fresh("post_activation_corruption")
    def corrupt_and_audit() -> None:
        manager.admit(package)
        victim = next(iter(tensors))
        manager.corrupt_active_tensor(victim, flat_index=0, delta=10.0)
        total_chunks = sum(len(v) for v in manager._active.expected_chunk_hashes.values())  # experiment only
        ok, failures = manager.audit_round(total_chunks, rng=random.Random(SEED))
        if ok or not failures:
            raise RuntimeError("complete chunk sweep missed corruption")
    record("post_activation_corruption", False, corrupt_and_audit)

    return pd.DataFrame(rows)


def run_concurrency_experiment(
    root_dir: os.PathLike[str] | str,
    manager: SecureAdapterManager,
    authority: UpdateAuthority,
    witness: OnlineWitness,
    target_specs: Sequence[TargetSpec],
    base_model_id: str,
    base_model_hash: str,
    model_inputs: Mapping[str, torch.Tensor],
    n_requests: int = 12,
    repeats: int = 20,
    delay_seconds: float = 0.002,
    rank: int = 4,
    alpha: float = 8.0,
) -> pd.DataFrame:
    from concurrent.futures import ThreadPoolExecutor

    root = Path(root_dir)
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True)

    tensors_v1 = build_random_lora_tensors(target_specs, rank, SEED + 11)
    package_v1 = write_signed_package(
        root / "v1", authority, witness, manager.deployment_id,
        base_model_id, base_model_hash, "adapter", 1, rank, alpha,
        tensors_v1, target_specs,
    )
    manager.admit(package_v1, make_warmup(manager, model_inputs))

    # Build v2 only after v1 is committed so its parent points to v1.
    tensors_v2 = build_random_lora_tensors(target_specs, rank, SEED + 12)
    package_v2 = write_signed_package(
        root / "v2", authority, witness, manager.deployment_id,
        base_model_id, base_model_hash, "adapter", 2, rank, alpha,
        tensors_v2, target_specs,
    )

    results: List[Dict[str, Any]] = []
    lock = threading.Lock()

    def worker(request_id: int) -> None:
        result = run_pinned_forward(manager, model_inputs, repeats, delay_seconds)
        result["request_id"] = request_id
        with lock:
            results.append(result)

    with ThreadPoolExecutor(max_workers=min(n_requests, 8)) as pool:
        first = [pool.submit(worker, i) for i in range(n_requests // 2)]
        time.sleep(delay_seconds * max(2, repeats // 4))
        manager.admit(package_v2, make_warmup(manager, model_inputs))
        second = [pool.submit(worker, i) for i in range(n_requests // 2, n_requests)]
        for future in first + second:
            future.result()

    frame = pd.DataFrame(results).sort_values("request_id").reset_index(drop=True)
    frame["trace_matches_pin"] = frame.apply(
        lambda row: row["observed_hashes"] in ([], [row["pinned_manifest_hash"]]), axis=1
    )
    return frame


def benchmark_forward_latency(
    manager: SecureAdapterManager,
    model_inputs: Mapping[str, torch.Tensor],
    trials_per_config: int = 30,
    warmup_runs: int = 10,
    seed: int = SEED,
) -> Tuple[pd.DataFrame, Dict[str, float]]:

    rng = random.Random(seed)
    with manager._lock:
        active = manager._active
    if active is None:
        raise RuntimeError("activate an adapter before benchmarking")

    for runtime in (None, active):
        for _ in range(warmup_runs):
            with torch.inference_mode(), manager.use_runtime(runtime):
                manager.model(**model_inputs)
        synchronize(manager.device)

    schedule = ["no_adapter"] * trials_per_config + ["active_adapter"] * trials_per_config
    rng.shuffle(schedule)
    rows: List[Dict[str, Any]] = []
    for index, config in enumerate(schedule):
        runtime = None if config == "no_adapter" else active
        synchronize(manager.device)
        started = time.perf_counter_ns()
        with torch.inference_mode(), manager.use_runtime(runtime):
            manager.model(**model_inputs)
        synchronize(manager.device)
        elapsed_ms = (time.perf_counter_ns() - started) / 1e6
        rows.append({"trial": index, "config": config, "latency_ms": elapsed_ms})

    frame = pd.DataFrame(rows)
    base = frame.loc[frame.config == "no_adapter", "latency_ms"].to_numpy()
    adapted = frame.loc[frame.config == "active_adapter", "latency_ms"].to_numpy()
    overhead = np.median(adapted) / np.median(base) - 1.0
    ci_low, ci_high = bootstrap_ratio_ci(adapted, base)
    summary = {
        "baseline_median_ms": float(np.median(base)),
        "adapter_median_ms": float(np.median(adapted)),
        "baseline_p95_ms": percentile(base, 95),
        "adapter_p95_ms": percentile(adapted, 95),
        "median_overhead_percent": float(overhead * 100.0),
        "overhead_ci95_low_percent": float(ci_low * 100.0),
        "overhead_ci95_high_percent": float(ci_high * 100.0),
    }
    return frame, summary


def demonstrate_torn_inplace_hotswap(
    device: torch.device,
    in_features: int = 32,
    out_features: int = 24,
    rank: int = 4,
    tokens: int = 8,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:

    generator = torch.Generator(device="cpu").manual_seed(SEED + 900)
    x = torch.randn(tokens, in_features, generator=generator, dtype=torch.float32).to(device)
    a1 = torch.randn(rank, in_features, generator=generator, dtype=torch.float32).to(device)
    b1 = torch.randn(out_features, rank, generator=generator, dtype=torch.float32).to(device)
    a2 = torch.randn(rank, in_features, generator=generator, dtype=torch.float32).to(device)
    b2 = torch.randn(out_features, rank, generator=generator, dtype=torch.float32).to(device)

    def apply(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        return F.linear(F.linear(x, a), b)

    with torch.inference_mode():
        references = {
            "v1": apply(a1, b1),
            "v2": apply(a2, b2),
            "hybrid_A1_B2": apply(a1, b2),
            "hybrid_A2_B1": apply(a2, b1),
        }
    synchronize(device)

    unsafe_a = a1.clone()
    unsafe_b = b1.clone()
    hidden_ready = threading.Event()
    update_complete = threading.Event()
    unsafe_result: Dict[str, torch.Tensor] = {}

    def unsafe_request() -> None:
        with torch.inference_mode():
            hidden = F.linear(x, unsafe_a)
            synchronize(device)
            hidden_ready.set()
            if not update_complete.wait(timeout=10):
                raise RuntimeError("timed out waiting for unsafe update")
            unsafe_result["output"] = F.linear(hidden, unsafe_b)
            synchronize(device)

    request_thread = threading.Thread(target=unsafe_request, daemon=True)
    request_thread.start()
    if not hidden_ready.wait(timeout=10):
        raise RuntimeError("timed out waiting for unsafe request")
    with torch.no_grad():
        unsafe_a.copy_(a2)
        synchronize(device)
        unsafe_b.copy_(b2)
        synchronize(device)
    update_complete.set()
    request_thread.join(timeout=10)
    if request_thread.is_alive():
        raise RuntimeError("unsafe request did not finish")

    runtime_v1 = {"version": 1, "A": a1, "B": b1}
    runtime_v2 = {"version": 2, "A": a2, "B": b2}
    active = {"runtime": runtime_v1}
    protected_hidden_ready = threading.Event()
    protected_update_complete = threading.Event()
    protected_result: Dict[str, Any] = {}

    def protected_request() -> None:
        pinned = active["runtime"]
        with torch.inference_mode():
            hidden = F.linear(x, pinned["A"])
            synchronize(device)
            protected_hidden_ready.set()
            if not protected_update_complete.wait(timeout=10):
                raise RuntimeError("timed out waiting for protected update")
            protected_result["output"] = F.linear(hidden, pinned["B"])
            protected_result["pinned_version"] = pinned["version"]
            synchronize(device)

    protected_thread = threading.Thread(target=protected_request, daemon=True)
    protected_thread.start()
    if not protected_hidden_ready.wait(timeout=10):
        raise RuntimeError("timed out waiting for protected request")
    active["runtime"] = runtime_v2
    protected_update_complete.set()
    protected_thread.join(timeout=10)
    if protected_thread.is_alive():
        raise RuntimeError("protected request did not finish")

    def classify(output: torch.Tensor) -> Tuple[str, Dict[str, float]]:
        distances = {
            name: float(torch.max(torch.abs(output - reference)).item())
            for name, reference in references.items()
        }
        return min(distances, key=distances.get), distances

    unsafe_class, unsafe_distances = classify(unsafe_result["output"])
    protected_class, protected_distances = classify(protected_result["output"])
    tolerance = 1e-5
    rows = [
        {
            "mode": "unsafe_in_place",
            "request_started_version": 1,
            "active_version_after_swap": 2,
            "observed_result": unsafe_class,
            "is_torn": unsafe_class.startswith("hybrid_") and unsafe_distances[unsafe_class] <= tolerance,
            **{f"max_abs_error_to_{k}": v for k, v in unsafe_distances.items()},
        },
        {
            "mode": "protected_pointer_pin",
            "request_started_version": int(protected_result["pinned_version"]),
            "active_version_after_swap": int(active["runtime"]["version"]),
            "observed_result": protected_class,
            "is_torn": protected_class.startswith("hybrid_") and protected_distances[protected_class] <= tolerance,
            **{f"max_abs_error_to_{k}": v for k, v in protected_distances.items()},
        },
    ]
    frame = pd.DataFrame(rows)
    summary = {
        "unsafe_torn_failure_reproduced": bool(frame.loc[frame["mode"] == "unsafe_in_place", "is_torn"].iloc[0]),
        "unsafe_observed_result": unsafe_class,
        "protected_observed_result": protected_class,
        "protected_prevented_torn_update": bool(protected_class == "v1" and protected_distances["v1"] <= tolerance),
        "schedule": [
            "request computes hidden state with A_v1",
            "updater copies A_v2 then B_v2 in place",
            "request resumes and reads B_v2",
        ],
    }
    return frame, summary


def benchmark_request_pinning_overhead(
    manager: SecureAdapterManager,
    model_inputs: Mapping[str, torch.Tensor],
    trials_per_config: int = 50,
    warmup_runs: int = 10,
    seed: int = SEED + 901,
) -> Tuple[pd.DataFrame, Dict[str, float]]:

    rng = random.Random(seed)
    with manager._lock:
        active = manager._active
    if active is None:
        raise RuntimeError("activate an adapter before benchmarking pinning overhead")

    for config in ("ordinary_active_lora", "protected_request_pin"):
        for _ in range(warmup_runs):
            context = manager.use_runtime(active, trace=False) if config == "ordinary_active_lora" else manager.pin_active(trace=False)
            with torch.inference_mode(), context:
                manager.model(**model_inputs)
        synchronize(manager.device)

    schedule = ["ordinary_active_lora"] * trials_per_config + ["protected_request_pin"] * trials_per_config
    rng.shuffle(schedule)
    rows: List[Dict[str, Any]] = []
    for index, config in enumerate(schedule):
        context = manager.use_runtime(active, trace=False) if config == "ordinary_active_lora" else manager.pin_active(trace=False)
        synchronize(manager.device)
        started = time.perf_counter_ns()
        with torch.inference_mode(), context:
            manager.model(**model_inputs)
        synchronize(manager.device)
        rows.append({
            "trial": index,
            "config": config,
            "latency_ms": (time.perf_counter_ns() - started) / 1e6,
        })

    frame = pd.DataFrame(rows)
    baseline = frame.loc[frame.config == "ordinary_active_lora", "latency_ms"].to_numpy()
    protected = frame.loc[frame.config == "protected_request_pin", "latency_ms"].to_numpy()
    overhead_ratio = np.median(protected) / np.median(baseline) - 1.0
    ci_low, ci_high = bootstrap_ratio_ci(protected, baseline)
    summary = {
        "ordinary_active_lora_median_ms": float(np.median(baseline)),
        "protected_request_pin_median_ms": float(np.median(protected)),
        "ordinary_active_lora_p95_ms": percentile(baseline, 95),
        "protected_request_pin_p95_ms": percentile(protected, 95),
        "median_overhead_ms": float(np.median(protected) - np.median(baseline)),
        "median_overhead_percent": float(overhead_ratio * 100.0),
        "overhead_ci95_low_percent": float(ci_low * 100.0),
        "overhead_ci95_high_percent": float(ci_high * 100.0),
        "trials_per_config": int(trials_per_config),
    }
    return frame, summary

def detection_probability_after_rounds(total_chunks: int, sampled_per_round: int, rounds: int, corrupt_chunks: int = 1) -> float:
    m = int(total_chunks)
    q = min(int(sampled_per_round), m)
    c = min(int(corrupt_chunks), m)
    if m <= 0 or q <= 0 or c <= 0:
        return 0.0
    if q > m - c:
        miss_one_round = 0.0
    else:
        miss_one_round = math.comb(m - c, q) / math.comb(m, q)
    return 1.0 - miss_one_round ** int(rounds)


def load_huggingface_demo(
    model_id: str = "sshleifer/tiny-gpt2",
    target_suffixes: Sequence[str] = ("c_attn", "c_proj"),
    max_target_modules: int = 4,
    max_length: int = 32,
) -> Dict[str, Any]:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_dtype = torch.float16 if device.type == "cuda" else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=load_dtype)
    except TypeError:
        # Transformers 4.x uses the legacy keyword.
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=load_dtype)
    model.eval().to(device)
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    base_hash = hash_model_state(model)
    specs = discover_lora_targets(model, target_suffixes, max_modules=max_target_modules)
    inject_versioned_lora(model, specs)

    encoded = tokenizer(
        "Secure adapter updates should never expose partially verified weights.",
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )
    model_inputs = {name: tensor.to(device) for name, tensor in encoded.items()}
    return {
        "model": model,
        "tokenizer": tokenizer,
        "device": device,
        "base_model_hash": base_hash,
        "target_specs": specs,
        "model_inputs": model_inputs,
    }


def run_colab_experiment(
    output_root: os.PathLike[str] | str = "/content/secure_atomic_lora_results",
    model_id: str = "sshleifer/tiny-gpt2",
    target_suffixes: Sequence[str] = ("c_attn", "c_proj"),
    max_target_modules: int = 4,
) -> Dict[str, Any]:
    output = Path(output_root)
    if output.exists():
        shutil.rmtree(output)
    output.mkdir(parents=True)

    setup = load_huggingface_demo(
        model_id=model_id,
        target_suffixes=target_suffixes,
        max_target_modules=max_target_modules,
    )
    model = setup["model"]
    device = setup["device"]
    specs = setup["target_specs"]
    base_hash = setup["base_model_hash"]
    model_inputs = setup["model_inputs"]

    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(device)}")
    print(f"Model: {model_id}")
    print(f"Base hash: {base_hash}")
    print("LoRA targets:")
    for spec in specs:
        print(f"  {spec.module_name}: {spec.in_features} -> {spec.out_features}")

    authority = UpdateAuthority()
    witness = OnlineWitness(authority.public_key, authority.key_id)
    deployment_id = f"colab-demo-{uuid.uuid4().hex[:8]}"
    manager = SecureAdapterManager(
        model=model,
        target_specs=specs,
        deployment_id=deployment_id,
        base_model_id=model_id,
        base_model_hash=base_hash,
        authority_public_key=authority.public_key,
        signing_key_id=authority.key_id,
        witness=witness,
        device=device,
        chunk_bytes=4096,
    )


    tensors = build_random_lora_tensors(specs, rank=4, seed=SEED + 100, dtype=torch.float32)
    package = write_signed_package(
        output / "main_package_v1",
        authority, witness, deployment_id, model_id, base_hash,
        adapter_id="research-adapter", adapter_version=1,
        rank=4, alpha=8.0, tensors=tensors, target_specs=specs,
    )
    admission_started = time.perf_counter()
    runtime = manager.admit(package, make_warmup(manager, model_inputs))
    synchronize(device)
    admission_ms = (time.perf_counter() - admission_started) * 1000.0
    print(f"Admitted version {runtime.version} in {admission_ms:.3f} ms")

    attack_frame = run_attack_suite(
        output / "attacks", model, specs, model_id, base_hash, device, model_inputs
    )
    attack_frame.to_csv(output / "attack_results.csv", index=False)
    print("\nAttack results:")
    print(attack_frame[["attack", "pass", "detail"]].to_string(index=False))


    concurrency_authority = UpdateAuthority()
    concurrency_witness = OnlineWitness(concurrency_authority.public_key, concurrency_authority.key_id)
    concurrency_manager = SecureAdapterManager(
        model=model,
        target_specs=specs,
        deployment_id=f"concurrency-{uuid.uuid4().hex[:8]}",
        base_model_id=model_id,
        base_model_hash=base_hash,
        authority_public_key=concurrency_authority.public_key,
        signing_key_id=concurrency_authority.key_id,
        witness=concurrency_witness,
        device=device,
        chunk_bytes=4096,
    )
    concurrency_frame = run_concurrency_experiment(
        output / "concurrency",
        concurrency_manager,
        concurrency_authority,
        concurrency_witness,
        specs,
        model_id,
        base_hash,
        model_inputs,
        n_requests=12,
        repeats=10 if device.type == "cpu" else 20,
    )
    concurrency_frame.to_csv(output / "concurrency_results.csv", index=False)
    print("\nConcurrency results:")
    print(concurrency_frame[["request_id", "pinned_version", "single_version", "trace_matches_pin"]].to_string(index=False))

    torn_frame, torn_summary = demonstrate_torn_inplace_hotswap(device)
    torn_frame.to_csv(output / "torn_update_results.csv", index=False)
    (output / "torn_update_summary.json").write_text(
        json.dumps(torn_summary, indent=2), encoding="utf-8"
    )
    print("\nDeterministic torn-update experiment:")
    print(torn_frame.to_string(index=False))
    print(json.dumps(torn_summary, indent=2))

    pinning_frame, pinning_summary = benchmark_request_pinning_overhead(
        manager, model_inputs,
        trials_per_config=40 if device.type == "cpu" else 100,
        warmup_runs=5 if device.type == "cpu" else 15,
    )
    pinning_frame.to_csv(output / "pinning_overhead_trials.csv", index=False)
    (output / "pinning_overhead_summary.json").write_text(
        json.dumps(pinning_summary, indent=2), encoding="utf-8"
    )
    print("\nRequest-pinning overhead:")
    print(json.dumps(pinning_summary, indent=2))

    latency_frame, latency_summary = benchmark_forward_latency(
        manager, model_inputs,
        trials_per_config=20 if device.type == "cpu" else 40,
        warmup_runs=5 if device.type == "cpu" else 10,
    )
    latency_frame.to_csv(output / "latency_trials.csv", index=False)
    (output / "latency_summary.json").write_text(
        json.dumps(latency_summary, indent=2), encoding="utf-8"
    )
    print("\nLatency summary:")
    print(json.dumps(latency_summary, indent=2))

    all_attacks_pass = bool(attack_frame["pass"].all())
    all_requests_consistent = bool(
        concurrency_frame["single_version"].all() and concurrency_frame["trace_matches_pin"].all()
    )
    summary = {
        "model_id": model_id,
        "device": str(device),
        "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "base_model_hash": base_hash,
        "deployment_id": deployment_id,
        "admission_ms": admission_ms,
        "all_attack_tests_pass": all_attacks_pass,
        "all_requests_single_version": all_requests_consistent,
        "torn_update": torn_summary,
        "pinning_overhead": pinning_summary,
        "latency": latency_summary,
        "output_root": str(output),
    }
    (output / "experiment_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return {
        "summary": summary,
        "attack_results": attack_frame,
        "concurrency_results": concurrency_frame,
        "torn_update_results": torn_frame,
        "pinning_overhead_trials": pinning_frame,
        "latency_trials": latency_frame,
        "manager": manager,
        "authority": authority,
        "witness": witness,
        "setup": setup,
    }


if __name__ == "__main__":
    print(
        "Import this file in Colab, or call run_colab_experiment() after installing "
        "transformers, safetensors, cryptography, pandas, and numpy."
    )


Writing secure_atomic_lora_colab.py


In [ ]:
import importlib
import secure_atomic_lora_colab as secure_lora
importlib.reload(secure_lora)


MODEL_ID = "sshleifer/tiny-gpt2"
TARGET_SUFFIXES = ("c_attn", "c_proj")
MAX_TARGET_MODULES = 4

experiment = secure_lora.run_colab_experiment(
    output_root="/content/secure_atomic_lora_results",
    model_id=MODEL_ID,
    target_suffixes=TARGET_SUFFIXES,
    max_target_modules=MAX_TARGET_MODULES,
)
experiment["summary"]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Device: cuda
GPU: Tesla T4
Model: sshleifer/tiny-gpt2
Base hash: e0e26b75975c098e0b2e62d53982309d3399b72ba04774f129ecd5b03304e95e
LoRA targets:
  transformer.h.0.attn.c_attn: 2 -> 6
  transformer.h.0.attn.c_proj: 2 -> 2
  transformer.h.0.mlp.c_proj: 8 -> 2
  transformer.h.1.attn.c_attn: 2 -> 6
Admitted version 1 in 1899.508 ms

Attack results:
                       attack  pass                                                                                                                                                                                                                                                                                                                                                                         detail
                valid_package  True                                                                                                                                                                                                                        

/content/secure_atomic_lora_colab.py:1427: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  hidden = F.linear(x, unsafe_a)



Request-pinning overhead:
{
  "ordinary_active_lora_median_ms": 3.038434,
  "protected_request_pin_median_ms": 3.038471,
  "ordinary_active_lora_p95_ms": 3.6944414999999986,
  "protected_request_pin_p95_ms": 3.3717441999999997,
  "median_overhead_ms": 3.6999999999842714e-05,
  "median_overhead_percent": 0.001217732555636175,
  "overhead_ci95_low_percent": -1.0862242807462088,
  "overhead_ci95_high_percent": 1.7935413789001813,
  "trials_per_config": 100
}

Latency summary:
{
  "baseline_median_ms": 2.4443764999999997,
  "adapter_median_ms": 3.0081545,
  "baseline_p95_ms": 2.963446749999998,
  "adapter_p95_ms": 3.4195694499999996,
  "median_overhead_percent": 23.064286536873514,
  "overhead_ci95_low_percent": 16.61176143417831,
  "overhead_ci95_high_percent": 28.8741672046677
}


{'model_id': 'sshleifer/tiny-gpt2',
 'device': 'cuda',
 'gpu': 'Tesla T4',
 'base_model_hash': 'e0e26b75975c098e0b2e62d53982309d3399b72ba04774f129ecd5b03304e95e',
 'deployment_id': 'colab-demo-99b2c792',
 'admission_ms': 1899.5084459999987,
 'all_attack_tests_pass': True,
 'all_requests_single_version': True,
 'torn_update': {'unsafe_torn_failure_reproduced': True,
  'unsafe_observed_result': 'hybrid_A1_B2',
  'protected_observed_result': 'v1',
  'protected_prevented_torn_update': True,
  'schedule': ['request computes hidden state with A_v1',
   'updater copies A_v2 then B_v2 in place',
   'request resumes and reads B_v2']},
 'pinning_overhead': {'ordinary_active_lora_median_ms': 3.038434,
  'protected_request_pin_median_ms': 3.038471,
  'ordinary_active_lora_p95_ms': 3.6944414999999986,
  'protected_request_pin_p95_ms': 3.3717441999999997,
  'median_overhead_ms': 3.6999999999842714e-05,
  'median_overhead_percent': 0.001217732555636175,
  'overhead_ci95_low_percent': -1.0862242807462

In [ ]:
attack_results = experiment["attack_results"]
concurrency_results = experiment["concurrency_results"]
torn_update_results = experiment["torn_update_results"]
pinning_overhead_trials = experiment["pinning_overhead_trials"]
latency_trials = experiment["latency_trials"]

display(attack_results[["attack", "pass", "detail", "elapsed_ms"]])
display(concurrency_results[["request_id", "pinned_version", "single_version", "trace_matches_pin"]])

print("Deterministic torn-update result:")
display(torn_update_results)
print(experiment["summary"]["torn_update"])

print("Cost of request pinning, compared with the same active LoRA without snapshot pinning:")
display(pinning_overhead_trials.groupby("config")["latency_ms"].describe(percentiles=[0.5, 0.95]))
print(experiment["summary"]["pinning_overhead"])


,attack,pass,detail,elapsed_ms
0,valid_package,True,accepted,19.487585
1,forged_signature,True,SecurityError: invalid manifest signature,0.401792
2,tampered_bytes,True,SecurityError: Safetensors package hash mismatch,2.231728
3,extra_tensor,True,SecurityError: adapter tensor scope mismatch; ...,0.616852
4,missing_tensor,True,SecurityError: adapter tensor scope mismatch; ...,0.587831
5,dtype_substitution,True,SecurityError: tensor 'transformer.h.0.attn.c_...,1.700434
6,shape_substitution,True,SecurityError: tensor 'transformer.h.0.attn.c_...,0.583034
7,mix_and_match_after_signing,True,SecurityError: Safetensors package hash mismatch,1.123235
8,embedded_tensor_name_mismatch,True,SecurityError: tensor record for 'transformer....,0.642410
9,wrong_deployment,True,SecurityError: package belongs to a different ...,0.535773


,request_id,pinned_version,single_version,trace_matches_pin
0,0,1,True,True
1,1,1,True,True
2,2,1,True,True
3,3,1,True,True
4,4,1,True,True
5,5,1,True,True
6,6,2,True,True
7,7,2,True,True
8,8,2,True,True
9,9,2,True,True


Deterministic torn-update result:


,mode,request_started_version,active_version_after_swap,observed_result,is_torn,max_abs_error_to_v1,max_abs_error_to_v2,max_abs_error_to_hybrid_A1_B2,max_abs_error_to_hybrid_A2_B1
0,unsafe_in_place,1,2,hybrid_A1_B2,True,58.688366,87.407127,0.000000,52.341682
1,protected_pointer_pin,1,2,v1,False,0.000000,52.198143,58.688366,56.587524


{'unsafe_torn_failure_reproduced': True, 'unsafe_observed_result': 'hybrid_A1_B2', 'protected_observed_result': 'v1', 'protected_prevented_torn_update': True, 'schedule': ['request computes hidden state with A_v1', 'updater copies A_v2 then B_v2 in place', 'request resumes and reads B_v2']}
Cost of request pinning, compared with the same active LoRA without snapshot pinning:


,count,mean,std,min,50%,95%,max
config,,,,,,,
ordinary_active_lora,100.0,3.127763,0.382923,2.784112,3.038434,3.694441,5.283851
protected_request_pin,100.0,3.101601,0.294770,2.785153,3.038471,3.371744,4.995868


{'ordinary_active_lora_median_ms': 3.038434, 'protected_request_pin_median_ms': 3.038471, 'ordinary_active_lora_p95_ms': 3.6944414999999986, 'protected_request_pin_p95_ms': 3.3717441999999997, 'median_overhead_ms': 3.6999999999842714e-05, 'median_overhead_percent': 0.001217732555636175, 'overhead_ci95_low_percent': -1.0862242807462088, 'overhead_ci95_high_percent': 1.7935413789001813, 'trials_per_config': 100}


In [ ]:
%%writefile paper_grade_peft_eval.py


from __future__ import annotations

import concurrent.futures
import contextlib
import dataclasses
import importlib.metadata
import json
import os
import platform
import random
import shutil
import socket
import statistics
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Mapping, MutableMapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch


DEFAULT_SEED = 20260722


@dataclass(frozen=True)
class PaperEvalConfig:
    """Configuration for the real-PEFT evaluation suite."""

    model_id: str = "sshleifer/tiny-gpt2"
    revision: Optional[str] = None
    trust_remote_code: bool = False
    target_modules: Tuple[str, ...] = ("c_attn",)
    rank: int = 4
    alpha: float = 8.0
    prompt: str = "Request-consistent adapter updates prevent undeployed hybrid model states."
    max_length: int = 48
    max_new_tokens: int = 12
    dtype: str = "float32"
    device: Optional[str] = None
    seed: int = DEFAULT_SEED
    stress_requests: int = 40
    stress_updates: int = 24
    stress_workers: int = 4
    benchmark_trials: int = 30
    benchmark_warmups: int = 5
    compile_backend: str = "inductor"
    match_atol: float = 1e-5
    match_rtol: float = 1e-4
    require_reference_separation: bool = True


@dataclass(frozen=True)
class PEFTArtifacts:
    output_root: str
    model_id: str
    adapter_v1: str
    adapter_v2: str
    references_path: str
    metadata_path: str
    target_layer_name: str
    adapter_name: str = "default"

    def as_dict(self) -> Dict[str, Any]:
        return asdict(self)


def _jsonable(value: Any) -> Any:
    if dataclasses.is_dataclass(value):
        return asdict(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, torch.dtype):
        return str(value)
    if isinstance(value, torch.device):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Mapping):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    return value


def write_json(path: os.PathLike[str] | str, value: Any) -> None:
    Path(path).write_text(json.dumps(_jsonable(value), indent=2, sort_keys=True), encoding="utf-8")


def synchronize(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def percentile(values: Sequence[float], q: float) -> float:
    array = np.asarray(values, dtype=np.float64)
    if array.size == 0:
        return float("nan")
    return float(np.percentile(array, q))


def bootstrap_ratio_ci(
    numerator: Sequence[float],
    denominator: Sequence[float],
    *,
    samples: int = 2000,
    seed: int = DEFAULT_SEED + 71,
) -> Tuple[float, float]:
    num = np.asarray(numerator, dtype=np.float64)
    den = np.asarray(denominator, dtype=np.float64)
    if num.size == 0 or den.size == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    ratios = np.empty(samples, dtype=np.float64)
    for index in range(samples):
        num_sample = rng.choice(num, size=num.size, replace=True)
        den_sample = rng.choice(den, size=den.size, replace=True)
        ratios[index] = np.median(num_sample) / np.median(den_sample) - 1.0
    return float(np.percentile(ratios, 2.5)), float(np.percentile(ratios, 97.5))


def resolve_device(config: PaperEvalConfig) -> torch.device:
    if config.device:
        return torch.device(config.device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def resolve_dtype(name: str, device: torch.device) -> torch.dtype:
    normalized = name.lower().replace("torch.", "")
    mapping = {
        "float32": torch.float32,
        "fp32": torch.float32,
        "float16": torch.float16,
        "fp16": torch.float16,
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
    }
    if normalized not in mapping:
        raise ValueError(f"unsupported dtype: {name}")
    dtype = mapping[normalized]
    if device.type == "cpu" and dtype == torch.float16:
        return torch.float32
    return dtype


def package_version(name: str) -> Optional[str]:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def environment_snapshot() -> Dict[str, Any]:
    snapshot: Dict[str, Any] = {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_runtime": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "packages": {
            name: package_version(name)
            for name in [
                "transformers",
                "peft",
                "accelerate",
                "safetensors",
                "numpy",
                "pandas",
                "psutil",
                "vllm",
            ]
        },
    }
    if torch.cuda.is_available():
        snapshot.update(
            {
                "gpu_name": torch.cuda.get_device_name(0),
                "gpu_count": torch.cuda.device_count(),
                "gpu_capability": list(torch.cuda.get_device_capability(0)),
                "gpu_total_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
            }
        )
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=driver_version,name,memory.total", "--format=csv,noheader"],
            check=False,
            capture_output=True,
            text=True,
            timeout=10,
        )
        if result.returncode == 0:
            snapshot["nvidia_smi"] = result.stdout.strip()
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return snapshot


def process_rss_bytes() -> int:
    try:
        import psutil

        return int(psutil.Process(os.getpid()).memory_info().rss)
    except Exception:
        return -1


def memory_snapshot(device: torch.device) -> Dict[str, int]:
    result = {"rss_bytes": process_rss_bytes()}
    if device.type == "cuda":
        result.update(
            {
                "cuda_allocated_bytes": int(torch.cuda.memory_allocated(device)),
                "cuda_reserved_bytes": int(torch.cuda.memory_reserved(device)),
                "cuda_max_allocated_bytes": int(torch.cuda.max_memory_allocated(device)),
            }
        )
    return result


def _load_transformers_model_and_tokenizer(
    config: PaperEvalConfig,
) -> Tuple[Any, Any, torch.device, Dict[str, torch.Tensor]]:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = resolve_device(config)
    dtype = resolve_dtype(config.dtype, device)
    tokenizer = AutoTokenizer.from_pretrained(
        config.model_id, revision=config.revision, trust_remote_code=config.trust_remote_code
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs: Dict[str, Any] = {}
    if dtype is not None:
        kwargs["torch_dtype"] = dtype
    model = AutoModelForCausalLM.from_pretrained(
        config.model_id, revision=config.revision, trust_remote_code=config.trust_remote_code, **kwargs
    )
    model.eval().to(device)
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    encoded = tokenizer(
        config.prompt,
        return_tensors="pt",
        truncation=True,
        max_length=config.max_length,
    )
    inputs = {name: value.to(device) for name, value in encoded.items()}
    return model, tokenizer, device, inputs


def _iter_lora_layers(model: torch.nn.Module, adapter_name: str) -> List[Tuple[str, torch.nn.Module]]:
    result: List[Tuple[str, torch.nn.Module]] = []
    for name, module in model.named_modules():
        lora_a = getattr(module, "lora_A", None)
        lora_b = getattr(module, "lora_B", None)
        if lora_a is None or lora_b is None:
            continue
        try:
            if adapter_name in lora_a and adapter_name in lora_b:
                result.append((name, module))
        except TypeError:
            continue
    if not result:
        raise RuntimeError(f"no PEFT LoRA layers found for adapter {adapter_name!r}")
    return result


def _find_lora_layer(model: torch.nn.Module, name: str, adapter_name: str) -> torch.nn.Module:
    candidates = dict(_iter_lora_layers(model, adapter_name))
    if name not in candidates:
        available = "\n".join(sorted(candidates)[:20])
        raise KeyError(f"LoRA layer {name!r} not found. First available layers:\n{available}")
    return candidates[name]


def _state_bytes(state: Mapping[str, Mapping[str, torch.Tensor]]) -> int:
    total = 0
    for pair in state.values():
        for tensor in pair.values():
            total += tensor.numel() * tensor.element_size()
    return int(total)


def _make_version_states(
    model: torch.nn.Module,
    adapter_name: str,
    seed: int,
) -> Tuple[
    Dict[str, Dict[str, torch.Tensor]],
    Dict[str, Dict[str, torch.Tensor]],
    Dict[str, Dict[str, torch.Tensor]],
    Dict[str, Dict[str, torch.Tensor]],
    str,
]:
    layers = _iter_lora_layers(model, adapter_name)
    target_name = layers[0][0]
    generator = torch.Generator(device="cpu").manual_seed(seed)
    state_v1: Dict[str, Dict[str, torch.Tensor]] = {}
    state_v2: Dict[str, Dict[str, torch.Tensor]] = {}

    for name, module in layers:
        a_param = module.lora_A[adapter_name].weight
        b_param = module.lora_B[adapter_name].weight
        common_a = torch.randn(a_param.shape, generator=generator, dtype=torch.float32) * 0.08
        common_b = torch.randn(b_param.shape, generator=generator, dtype=torch.float32) * 0.08
        if name == target_name:
            a1 = torch.randn(a_param.shape, generator=generator, dtype=torch.float32) * 0.20
            b1 = torch.randn(b_param.shape, generator=generator, dtype=torch.float32) * 0.20
            a2 = torch.randn(a_param.shape, generator=generator, dtype=torch.float32) * 0.20
            b2 = torch.randn(b_param.shape, generator=generator, dtype=torch.float32) * 0.20
        else:
            # Other layers remain identical. This isolates the classification of
            # the forced race to one real PEFT layer.
            a1 = common_a.clone()
            b1 = common_b.clone()
            a2 = common_a.clone()
            b2 = common_b.clone()
        state_v1[name] = {"A": a1, "B": b1}
        state_v2[name] = {"A": a2, "B": b2}

    hybrid_a1_b2 = {
        name: {
            "A": state_v1[name]["A"].clone(),
            "B": state_v2[name]["B"].clone(),
        }
        for name in state_v1
    }
    hybrid_a2_b1 = {
        name: {
            "A": state_v2[name]["A"].clone(),
            "B": state_v1[name]["B"].clone(),
        }
        for name in state_v1
    }
    return state_v1, state_v2, hybrid_a1_b2, hybrid_a2_b1, target_name


def _apply_version_state(
    model: torch.nn.Module,
    adapter_name: str,
    state: Mapping[str, Mapping[str, torch.Tensor]],
) -> None:
    layers = dict(_iter_lora_layers(model, adapter_name))
    with torch.no_grad():
        for name, pair in state.items():
            if name not in layers:
                raise KeyError(f"adapter state contains unknown layer {name}")
            layer = layers[name]
            a_weight = layer.lora_A[adapter_name].weight
            b_weight = layer.lora_B[adapter_name].weight
            a_weight.copy_(pair["A"].to(device=a_weight.device, dtype=a_weight.dtype))
            b_weight.copy_(pair["B"].to(device=b_weight.device, dtype=b_weight.dtype))


def _forward_logits(model: torch.nn.Module, inputs: Mapping[str, torch.Tensor], **kwargs: Any) -> torch.Tensor:
    device = next(model.parameters()).device
    with torch.inference_mode():
        output = model(**inputs, **kwargs)
        logits = output.logits.detach().float().cpu()
    synchronize(device)
    return logits


def _classify_logits(
    observed: torch.Tensor,
    references: Mapping[str, torch.Tensor],
    atol: float,
    rtol: float = 0.0,
) -> Tuple[str, Dict[str, float], bool]:
    """Classify an output only when exactly one reference is within tolerance.

    A nearest-neighbor label alone can be misleading when two reference outputs
    are numerically too close. The returned boolean therefore requires a unique
    tolerance match, not merely the smallest distance.
    """
    observed_cpu = observed.float().cpu()
    distances = {
        name: float(torch.max(torch.abs(observed_cpu - reference.float().cpu())).item())
        for name, reference in references.items()
    }
    thresholds = {
        name: float(atol + rtol * torch.max(torch.abs(reference.float().cpu())).item())
        for name, reference in references.items()
    }
    closest = min(distances, key=distances.get)
    within = [name for name, distance in distances.items() if distance <= thresholds[name]]
    return closest, distances, bool(len(within) == 1 and within[0] == closest)


def _reference_separation_report(
    references: Mapping[str, torch.Tensor],
    atol: float,
    rtol: float,
) -> Dict[str, Any]:
    names = sorted(references)
    pairwise: Dict[str, float] = {}
    for i, left in enumerate(names):
        for right in names[i + 1 :]:
            distance = float(
                torch.max(
                    torch.abs(references[left].float().cpu() - references[right].float().cpu())
                ).item()
            )
            pairwise[f"{left}__vs__{right}"] = distance
    thresholds = {
        name: float(atol + rtol * torch.max(torch.abs(value.float().cpu())).item())
        for name, value in references.items()
    }
    minimum_distance = min(pairwise.values()) if pairwise else float("inf")
    maximum_threshold = max(thresholds.values()) if thresholds else float(atol)
    return {
        "pairwise_max_abs_distances": pairwise,
        "minimum_pairwise_distance": minimum_distance,
        "maximum_match_threshold": maximum_threshold,
        "separable": bool(minimum_distance > 2.0 * maximum_threshold),
    }


def _save_adapter(model: Any, path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path, safe_serialization=True)


def build_real_peft_artifacts(
    output_root: os.PathLike[str] | str,
    config: PaperEvalConfig,
) -> PEFTArtifacts:
    """Create deterministic PEFT v1/v2 adapters and numerical references."""
    from peft import LoraConfig, TaskType, get_peft_model

    output = Path(output_root)
    output.mkdir(parents=True, exist_ok=True)
    model, tokenizer, device, inputs = _load_transformers_model_and_tokenizer(config)
    lora_config = LoraConfig(
        r=config.rank,
        lora_alpha=config.alpha,
        lora_dropout=0.0,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=list(config.target_modules),
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.eval()
    adapter_name = "default"

    state_v1, state_v2, hybrid_a1_b2, hybrid_a2_b1, target_name = _make_version_states(
        peft_model, adapter_name, config.seed + 1000
    )

    adapter_v1 = output / "adapter_v1"
    adapter_v2 = output / "adapter_v2"
    _apply_version_state(peft_model, adapter_name, state_v1)
    synchronize(device)
    reference_v1 = _forward_logits(peft_model, inputs)
    _save_adapter(peft_model, adapter_v1)

    _apply_version_state(peft_model, adapter_name, state_v2)
    synchronize(device)
    reference_v2 = _forward_logits(peft_model, inputs)
    _save_adapter(peft_model, adapter_v2)

    _apply_version_state(peft_model, adapter_name, hybrid_a1_b2)
    synchronize(device)
    reference_a1_b2 = _forward_logits(peft_model, inputs)

    _apply_version_state(peft_model, adapter_name, hybrid_a2_b1)
    synchronize(device)
    reference_a2_b1 = _forward_logits(peft_model, inputs)

    references = {
        "v1": reference_v1,
        "v2": reference_v2,
        "hybrid_A1_B2": reference_a1_b2,
        "hybrid_A2_B1": reference_a2_b1,
    }
    separation = _reference_separation_report(
        references, config.match_atol, config.match_rtol
    )
    if config.require_reference_separation and not separation["separable"]:
        raise RuntimeError(
            "The generated v1/v2/hybrid references are not numerically separable at the "
            "configured tolerance. Increase adapter magnitude or use a more precise dtype."
        )
    references_path = output / "reference_logits.pt"
    torch.save(references, references_path)

    metadata = {
        "config": asdict(config),
        "model_id": config.model_id,
        "adapter_name": adapter_name,
        "target_layer_name": target_name,
        "all_lora_layers": [name for name, _ in _iter_lora_layers(peft_model, adapter_name)],
        "adapter_tensor_bytes_per_version": _state_bytes(state_v1),
        "input_shapes": {name: list(tensor.shape) for name, tensor in inputs.items()},
        "tokenizer_class": tokenizer.__class__.__name__,
        "device": str(device),
        "dtype": str(next(peft_model.parameters()).dtype),
        "resolved_model_commit": getattr(getattr(peft_model, "config", None), "_commit_hash", None),
        "reference_separation": separation,
    }
    metadata_path = output / "artifacts_metadata.json"
    write_json(metadata_path, metadata)

    del peft_model, model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return PEFTArtifacts(
        output_root=str(output),
        model_id=config.model_id,
        adapter_v1=str(adapter_v1),
        adapter_v2=str(adapter_v2),
        references_path=str(references_path),
        metadata_path=str(metadata_path),
        target_layer_name=target_name,
        adapter_name=adapter_name,
    )


def _load_references(artifacts: PEFTArtifacts) -> Dict[str, torch.Tensor]:
    loaded = torch.load(artifacts.references_path, map_location="cpu")
    return {str(name): tensor.float().cpu() for name, tensor in loaded.items()}


def _load_peft_model(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    *,
    adapter_path: Optional[str] = None,
    adapter_name: Optional[str] = None,
) -> Tuple[Any, Any, torch.device, Dict[str, torch.Tensor]]:
    from peft import PeftModel

    base_model, tokenizer, device, inputs = _load_transformers_model_and_tokenizer(config)
    model = PeftModel.from_pretrained(
        base_model,
        adapter_path or artifacts.adapter_v1,
        adapter_name=adapter_name or artifacts.adapter_name,
        is_trainable=False,
    )
    model.eval()
    return model, tokenizer, device, inputs


def _hotswap_adapter(model: Any, adapter_path: str, adapter_name: str, device: torch.device) -> None:
    try:
        from peft.utils.hotswap import hotswap_adapter

        hotswap_adapter(model, adapter_path, adapter_name=adapter_name, torch_device=str(device))
    except (ImportError, AttributeError):
        # Newer Transformers-integrated PEFT exposes hotswap through load_adapter.
        model.load_adapter(adapter_path, adapter_name=adapter_name, hotswap=True)


@dataclass(frozen=True)
class PeftVersionLease:
    adapter_name: str
    logical_version: int


class DoubleBufferedPeftRouter:
    """RCU-style two-slot router built on real PEFT named adapters.

    Requests pin one slot for their entire lifetime. Updates hotswap only the
    inactive slot after its lease count reaches zero, validate it, and publish
    it with one small pointer change. The old slot is not reused until all
    requests that pinned it have completed.
    """

    SLOT_NAMES = ("__atomic_slot_0", "__atomic_slot_1")

    def __init__(
        self,
        model: Any,
        device: torch.device,
        *,
        active_slot: str = SLOT_NAMES[0],
        active_version: int = 1,
    ) -> None:
        if active_slot not in self.SLOT_NAMES:
            raise ValueError(f"unknown active slot: {active_slot}")
        self.model = model
        self.device = device
        self._active_slot = active_slot
        self._slot_versions: Dict[str, Optional[int]] = {
            self.SLOT_NAMES[0]: active_version if active_slot == self.SLOT_NAMES[0] else None,
            self.SLOT_NAMES[1]: active_version if active_slot == self.SLOT_NAMES[1] else None,
        }
        self._leases = {name: 0 for name in self.SLOT_NAMES}
        self._condition = threading.Condition(threading.RLock())
        self._update_lock = threading.Lock()

    @property
    def active_version(self) -> int:
        with self._condition:
            value = self._slot_versions[self._active_slot]
            if value is None:
                raise RuntimeError("active slot has no logical version")
            return int(value)

    def snapshot(self) -> Dict[str, Any]:
        with self._condition:
            return {
                "active_slot": self._active_slot,
                "active_version": self._slot_versions[self._active_slot],
                "slot_versions": dict(self._slot_versions),
                "leases": dict(self._leases),
            }

    @contextlib.contextmanager
    def pin(self) -> Iterable[PeftVersionLease]:
        with self._condition:
            slot = self._active_slot
            version = self._slot_versions[slot]
            if version is None:
                raise RuntimeError("cannot pin an uninitialized PEFT slot")
            self._leases[slot] += 1
        try:
            yield PeftVersionLease(slot, int(version))
        finally:
            with self._condition:
                self._leases[slot] -= 1
                if self._leases[slot] < 0:
                    raise RuntimeError("negative PEFT slot lease count")
                self._condition.notify_all()

    def forward_logits(
        self, inputs: Mapping[str, torch.Tensor]
    ) -> Tuple[torch.Tensor, PeftVersionLease]:
        batch_size = int(inputs["input_ids"].shape[0])
        with self.pin() as lease:
            logits = _forward_logits(
                self.model,
                inputs,
                adapter_names=[lease.adapter_name] * batch_size,
            )
            return logits, lease

    def publish(
        self,
        adapter_path: str,
        *,
        logical_version: int,
        validator: Optional[Callable[[str], None]] = None,
        timeout_s: float = 120.0,
    ) -> Dict[str, Any]:
        if logical_version <= 0:
            raise ValueError("logical_version must be positive")
        deadline = time.monotonic() + timeout_s
        with self._update_lock:
            with self._condition:
                active = self._active_slot
                inactive = self.SLOT_NAMES[1] if active == self.SLOT_NAMES[0] else self.SLOT_NAMES[0]
                wait_started = time.perf_counter_ns()
                while self._leases[inactive] > 0:
                    remaining = deadline - time.monotonic()
                    if remaining <= 0:
                        raise TimeoutError(
                            f"timed out waiting for inactive slot {inactive} to become quiescent"
                        )
                    self._condition.wait(timeout=remaining)
                quiescence_wait_ms = (time.perf_counter_ns() - wait_started) / 1e6

            stage_started = time.perf_counter_ns()
            _hotswap_adapter(self.model, adapter_path, inactive, self.device)
            synchronize(self.device)
            stage_ms = (time.perf_counter_ns() - stage_started) / 1e6

            validation_ms = 0.0
            if validator is not None:
                validation_started = time.perf_counter_ns()
                validator(inactive)
                synchronize(self.device)
                validation_ms = (time.perf_counter_ns() - validation_started) / 1e6

            with self._condition:
                publish_started = time.perf_counter_ns()
                previous_slot = self._active_slot
                previous_version = self._slot_versions[previous_slot]
                self._slot_versions[inactive] = int(logical_version)
                self._active_slot = inactive
                publish_us = (time.perf_counter_ns() - publish_started) / 1e3
                self._condition.notify_all()

            return {
                "previous_slot": previous_slot,
                "previous_version": previous_version,
                "published_slot": inactive,
                "published_version": int(logical_version),
                "quiescence_wait_ms": quiescence_wait_ms,
                "stage_hotswap_ms": stage_ms,
                "validation_ms": validation_ms,
                "pointer_publish_us": publish_us,
            }


def _load_double_buffer_router(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
) -> Tuple[DoubleBufferedPeftRouter, Any, torch.device, Dict[str, torch.Tensor]]:
    slot0, slot1 = DoubleBufferedPeftRouter.SLOT_NAMES
    model, tokenizer, device, inputs = _load_peft_model(
        artifacts, config, adapter_path=artifacts.adapter_v1, adapter_name=slot0
    )
    model.load_adapter(artifacts.adapter_v1, adapter_name=slot1)
    model.set_adapter(slot0)
    router = DoubleBufferedPeftRouter(
        model, device, active_slot=slot0, active_version=1
    )
    return router, tokenizer, device, inputs


def run_double_buffered_peft_update(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Demonstrate a real-PEFT safe update and quiescent slot reuse."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    references = _load_references(artifacts)
    router, _, device, inputs = _load_double_buffer_router(artifacts, config)
    slot0, slot1 = DoubleBufferedPeftRouter.SLOT_NAMES
    target_layer = _find_lora_layer(router.model, artifacts.target_layer_name, slot0)

    paused_after_a = threading.Event()
    resume_request = threading.Event()
    request_result: Dict[str, Any] = {}

    def pause_hook(_module: torch.nn.Module, _args: Tuple[Any, ...], _output: Any) -> None:
        synchronize(device)
        paused_after_a.set()
        if not resume_request.wait(timeout=120):
            raise TimeoutError("double-buffer request timed out while pinned to the old slot")

    hook = target_layer.lora_A[slot0].register_forward_hook(pause_hook)

    def request_worker() -> None:
        try:
            logits, lease = router.forward_logits(inputs)
            request_result.update({"logits": logits, "lease": lease})
        except BaseException as exc:
            request_result["error"] = repr(exc)

    request_thread = threading.Thread(
        target=request_worker, name="double-buffer-old-request", daemon=True
    )
    request_thread.start()
    if not paused_after_a.wait(timeout=120):
        hook.remove()
        raise TimeoutError("double-buffer request did not reach the slot-0 lora_A hook")

    def validate_v2(slot: str) -> None:
        batch_size = int(inputs["input_ids"].shape[0])
        logits = _forward_logits(router.model, inputs, adapter_names=[slot] * batch_size)
        closest, _, matched = _classify_logits(
            logits, references, config.match_atol, config.match_rtol
        )
        if closest != "v2" or not matched:
            raise RuntimeError(f"staged slot failed v2 validation: {closest}")

    publish_v2 = router.publish(
        artifacts.adapter_v2, logical_version=2, validator=validate_v2
    )
    new_logits, new_lease = router.forward_logits(inputs)
    new_class, new_distances, new_match = _classify_logits(
        new_logits, references, config.match_atol, config.match_rtol
    )

    recycle_result: Dict[str, Any] = {}

    def validate_recycled_v1(slot: str) -> None:
        batch_size = int(inputs["input_ids"].shape[0])
        logits = _forward_logits(router.model, inputs, adapter_names=[slot] * batch_size)
        closest, _, matched = _classify_logits(
            logits, references, config.match_atol, config.match_rtol
        )
        if closest != "v1" or not matched:
            raise RuntimeError(f"recycled slot failed v1 validation: {closest}")

    def recycle_worker() -> None:
        try:
            recycle_result["metrics"] = router.publish(
                artifacts.adapter_v1, logical_version=3, validator=validate_recycled_v1
            )
        except BaseException as exc:
            recycle_result["error"] = repr(exc)

    recycle_thread = threading.Thread(
        target=recycle_worker, name="double-buffer-recycle", daemon=True
    )
    recycle_thread.start()
    time.sleep(0.05)
    recycle_blocked_while_old_request_live = recycle_thread.is_alive()

    resume_request.set()
    request_thread.join(timeout=120)
    recycle_thread.join(timeout=120)
    hook.remove()
    if request_thread.is_alive() or recycle_thread.is_alive():
        raise TimeoutError("double-buffer experiment did not finish")
    if "error" in request_result:
        raise RuntimeError(request_result["error"])
    if "error" in recycle_result:
        raise RuntimeError(recycle_result["error"])

    old_class, old_distances, old_match = _classify_logits(
        request_result["logits"], references, config.match_atol, config.match_rtol
    )
    old_lease: PeftVersionLease = request_result["lease"]
    recycle_metrics = recycle_result["metrics"]

    frame = pd.DataFrame(
        [
            {
                "request": "in_flight_before_publish",
                "pinned_slot": old_lease.adapter_name,
                "pinned_version": old_lease.logical_version,
                "observed_result": old_class,
                "within_tolerance": old_match,
                "single_version": bool(old_class == "v1" and old_match),
                **{f"max_abs_error_to_{name}": value for name, value in old_distances.items()},
            },
            {
                "request": "started_after_publish",
                "pinned_slot": new_lease.adapter_name,
                "pinned_version": new_lease.logical_version,
                "observed_result": new_class,
                "within_tolerance": new_match,
                "single_version": bool(new_class == "v2" and new_match),
                **{f"max_abs_error_to_{name}": value for name, value in new_distances.items()},
            },
        ]
    )
    summary = {
        "strategy": "real_peft_double_buffered_rcu",
        "safe": bool(
            old_class == "v1"
            and old_match
            and new_class == "v2"
            and new_match
            and recycle_blocked_while_old_request_live
        ),
        "in_flight_request_result": old_class,
        "post_publish_request_result": new_class,
        "recycle_blocked_while_old_request_live": recycle_blocked_while_old_request_live,
        "publish_v2": publish_v2,
        "recycle_after_quiescence": recycle_metrics,
        "final_router_state": router.snapshot(),
        "invariant": (
            "The updater mutates only a slot with zero leases; each request names its pinned slot "
            "at every PEFT layer; publication changes only the active-slot pointer."
        ),
    }
    frame.to_csv(output / "double_buffered_peft_update.csv", index=False)
    write_json(output / "double_buffered_peft_update.json", summary)
    return frame, summary


def benchmark_double_buffer_router_overhead(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Isolate lease/pointer overhead over the same PEFT adapter_names path."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    router, _, device, inputs = _load_double_buffer_router(artifacts, config)
    slot0 = DoubleBufferedPeftRouter.SLOT_NAMES[0]
    batch_size = int(inputs["input_ids"].shape[0])

    def run(mode: str) -> None:
        if mode == "adapter_names_only":
            _forward_logits(router.model, inputs, adapter_names=[slot0] * batch_size)
        else:
            router.forward_logits(inputs)

    for mode in ("adapter_names_only", "double_buffer_router"):
        for _ in range(config.benchmark_warmups):
            run(mode)

    schedule = ["adapter_names_only"] * config.benchmark_trials + [
        "double_buffer_router"
    ] * config.benchmark_trials
    random.Random(config.seed + 3201).shuffle(schedule)
    rows: List[Dict[str, Any]] = []
    for trial, mode in enumerate(schedule):
        synchronize(device)
        started = time.perf_counter_ns()
        run(mode)
        rows.append(
            {
                "trial": trial,
                "mode": mode,
                "latency_ms": (time.perf_counter_ns() - started) / 1e6,
            }
        )

    frame = pd.DataFrame(rows)
    baseline = frame.loc[frame["mode"] == "adapter_names_only", "latency_ms"].to_numpy()
    routed = frame.loc[frame["mode"] == "double_buffer_router", "latency_ms"].to_numpy()
    ratio = float(np.median(routed) / np.median(baseline) - 1.0)
    ci_low, ci_high = bootstrap_ratio_ci(routed, baseline, seed=config.seed + 3202)
    summary = {
        "adapter_names_only_median_ms": float(np.median(baseline)),
        "double_buffer_router_median_ms": float(np.median(routed)),
        "adapter_names_only_p95_ms": percentile(baseline, 95),
        "double_buffer_router_p95_ms": percentile(routed, 95),
        "median_router_overhead_percent": ratio * 100.0,
        "router_overhead_ci95_percent": [ci_low * 100.0, ci_high * 100.0],
        "trials_per_mode": config.benchmark_trials,
    }
    frame.to_csv(output / "double_buffer_router_overhead_trials.csv", index=False)
    write_json(output / "double_buffer_router_overhead_summary.json", summary)
    return frame, summary


def run_real_peft_torn_update(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Deterministically force A_v1/B_v2 inside a real PEFT LoRA layer."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    references = _load_references(artifacts)
    model, _, device, inputs = _load_peft_model(artifacts, config)
    target_layer = _find_lora_layer(model, artifacts.target_layer_name, artifacts.adapter_name)
    lora_a = target_layer.lora_A[artifacts.adapter_name]

    hidden_ready = threading.Event()
    resume_request = threading.Event()
    request_result: Dict[str, Any] = {}

    def pause_after_a(_module: torch.nn.Module, _args: Tuple[Any, ...], _output: Any) -> None:
        synchronize(device)
        hidden_ready.set()
        if not resume_request.wait(timeout=60):
            raise TimeoutError("request timed out while paused after PEFT lora_A")

    hook = lora_a.register_forward_hook(pause_after_a)

    def request_worker() -> None:
        try:
            request_result["logits"] = _forward_logits(model, inputs)
        except BaseException as exc:
            request_result["error"] = repr(exc)

    request_thread = threading.Thread(target=request_worker, name="peft-race-request", daemon=True)
    request_thread.start()
    if not hidden_ready.wait(timeout=60):
        hook.remove()
        raise TimeoutError("real PEFT request did not reach lora_A hook")

    swap_started = time.perf_counter_ns()
    _hotswap_adapter(model, artifacts.adapter_v2, artifacts.adapter_name, device)
    synchronize(device)
    swap_ms = (time.perf_counter_ns() - swap_started) / 1e6
    resume_request.set()
    request_thread.join(timeout=60)
    hook.remove()
    if request_thread.is_alive():
        raise TimeoutError("real PEFT request did not finish")
    if "error" in request_result:
        raise RuntimeError(request_result["error"])

    observed = request_result["logits"]
    closest, distances, matched = _classify_logits(observed, references, config.match_atol, config.match_rtol)
    post_swap = _forward_logits(model, inputs)
    post_closest, post_distances, post_matched = _classify_logits(post_swap, references, config.match_atol, config.match_rtol)

    row = {
        "experiment": "real_peft_hotswap",
        "selected_layer": artifacts.target_layer_name,
        "observed_result": closest,
        "within_tolerance": matched,
        "is_torn": closest.startswith("hybrid_") and matched,
        "hotswap_ms": swap_ms,
        **{f"max_abs_error_to_{name}": value for name, value in distances.items()},
    }
    frame = pd.DataFrame([row])
    summary = {
        "uses_real_peft_layer": True,
        "uses_real_peft_hotswap_api": True,
        "selected_layer": artifacts.target_layer_name,
        "observed_result": closest,
        "torn_update_reproduced": bool(closest == "hybrid_A1_B2" and matched),
        "post_swap_result": post_closest,
        "post_swap_matches_v2": bool(post_closest == "v2" and post_matched),
        "post_swap_distances": post_distances,
        "hotswap_ms": swap_ms,
        "schedule": [
            "real PEFT lora_A(default) computes with version 1",
            "forward hook pauses the request after lora_A",
            "PEFT hotswap_adapter replaces the adapter in place with version 2",
            "request resumes through real PEFT lora_B(default)",
        ],
    }
    frame.to_csv(output / "real_peft_torn_update.csv", index=False)
    write_json(output / "real_peft_torn_update.json", summary)
    return frame, summary


def run_named_adapter_request_pin_test(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Show PEFT's explicit adapter_names path is request-scoped for a batch."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    references = _load_references(artifacts)
    model, _, device, inputs = _load_peft_model(
        artifacts, config, adapter_path=artifacts.adapter_v1, adapter_name="v1"
    )
    model.load_adapter(artifacts.adapter_v2, adapter_name="v2")
    model.set_adapter("v1")
    target_layer = _find_lora_layer(model, artifacts.target_layer_name, "v1")

    hidden_ready = threading.Event()
    resume_request = threading.Event()
    request_result: Dict[str, Any] = {}

    def pause_after_a(_module: torch.nn.Module, _args: Tuple[Any, ...], _output: Any) -> None:
        synchronize(device)
        hidden_ready.set()
        if not resume_request.wait(timeout=60):
            raise TimeoutError("named-adapter request timed out")

    hook = target_layer.lora_A["v1"].register_forward_hook(pause_after_a)

    def request_worker() -> None:
        try:
            request_result["logits"] = _forward_logits(model, inputs, adapter_names=["v1"])
        except BaseException as exc:
            request_result["error"] = repr(exc)

    thread = threading.Thread(target=request_worker, name="peft-named-adapter-request", daemon=True)
    thread.start()
    if not hidden_ready.wait(timeout=60):
        hook.remove()
        raise TimeoutError("named adapter request did not reach lora_A")
    model.set_adapter("v2")
    resume_request.set()
    thread.join(timeout=60)
    hook.remove()
    if thread.is_alive():
        raise TimeoutError("named adapter request did not finish")
    if "error" in request_result:
        raise RuntimeError(request_result["error"])

    closest, distances, matched = _classify_logits(
        request_result["logits"], references, config.match_atol, config.match_rtol
    )
    row = {
        "experiment": "peft_adapter_names_request_pin",
        "requested_adapter": "v1",
        "global_active_after_switch": "v2",
        "observed_result": closest,
        "within_tolerance": matched,
        "single_version": closest == "v1" and matched,
        **{f"max_abs_error_to_{name}": value for name, value in distances.items()},
    }
    frame = pd.DataFrame([row])
    summary = {
        "requested_adapter": "v1",
        "global_active_after_switch": "v2",
        "observed_result": closest,
        "request_remained_on_v1": bool(closest == "v1" and matched),
        "interpretation": (
            "Explicit per-sample adapter_names is a useful PEFT baseline. It avoids relying "
            "on a mutable global active adapter, but requires both versions to be resident."
        ),
    }
    frame.to_csv(output / "named_adapter_pin.csv", index=False)
    write_json(output / "named_adapter_pin.json", summary)
    return frame, summary


def run_full_model_mutex_test(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
    hold_ms: float = 50.0,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    references = _load_references(artifacts)
    model, _, device, inputs = _load_peft_model(artifacts, config)
    target_layer = _find_lora_layer(model, artifacts.target_layer_name, artifacts.adapter_name)
    mutex = threading.Lock()
    hidden_ready = threading.Event()
    resume_request = threading.Event()
    updater_attempted = threading.Event()
    request_result: Dict[str, Any] = {}
    update_result: Dict[str, Any] = {}

    def pause_after_a(_module: torch.nn.Module, _args: Tuple[Any, ...], _output: Any) -> None:
        synchronize(device)
        hidden_ready.set()
        if not resume_request.wait(timeout=60):
            raise TimeoutError("mutex request timed out")

    hook = target_layer.lora_A[artifacts.adapter_name].register_forward_hook(pause_after_a)

    def request_worker() -> None:
        started = time.perf_counter_ns()
        try:
            with mutex:
                request_result["logits"] = _forward_logits(model, inputs)
        except BaseException as exc:
            request_result["error"] = repr(exc)
        request_result["latency_ms"] = (time.perf_counter_ns() - started) / 1e6

    def update_worker() -> None:
        started = time.perf_counter_ns()
        updater_attempted.set()
        try:
            with mutex:
                acquired = time.perf_counter_ns()
                _hotswap_adapter(model, artifacts.adapter_v2, artifacts.adapter_name, device)
                synchronize(device)
            update_result["lock_wait_ms"] = (acquired - started) / 1e6
            update_result["total_ms"] = (time.perf_counter_ns() - started) / 1e6
        except BaseException as exc:
            update_result["error"] = repr(exc)

    request_thread = threading.Thread(target=request_worker, daemon=True)
    request_thread.start()
    if not hidden_ready.wait(timeout=60):
        hook.remove()
        raise TimeoutError("mutex request did not reach lora_A")
    updater_thread = threading.Thread(target=update_worker, daemon=True)
    updater_thread.start()
    updater_attempted.wait(timeout=10)
    time.sleep(max(0.0, hold_ms) / 1000.0)
    resume_request.set()
    request_thread.join(timeout=60)
    updater_thread.join(timeout=60)
    hook.remove()
    if request_thread.is_alive() or updater_thread.is_alive():
        raise TimeoutError("mutex experiment did not finish")
    if "error" in request_result:
        raise RuntimeError(request_result["error"])
    if "error" in update_result:
        raise RuntimeError(update_result["error"])

    closest, distances, matched = _classify_logits(
        request_result["logits"], references, config.match_atol, config.match_rtol
    )
    row = {
        "experiment": "full_model_mutex",
        "observed_result": closest,
        "within_tolerance": matched,
        "single_version": closest == "v1" and matched,
        "request_latency_ms": request_result["latency_ms"],
        "update_lock_wait_ms": update_result["lock_wait_ms"],
        "update_total_ms": update_result["total_ms"],
        **{f"max_abs_error_to_{name}": value for name, value in distances.items()},
    }
    frame = pd.DataFrame([row])
    summary = {
        "safe": bool(closest == "v1" and matched),
        "request_result": closest,
        "update_lock_wait_ms": update_result["lock_wait_ms"],
        "update_total_ms": update_result["total_ms"],
        "tradeoff": "The update is safe because it waits for the whole request-side critical section.",
    }
    frame.to_csv(output / "full_model_mutex.csv", index=False)
    write_json(output / "full_model_mutex.json", summary)
    return frame, summary


def benchmark_named_adapter_overhead(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Compare active-adapter inference with PEFT adapter_names inference."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    model, _, device, inputs = _load_peft_model(
        artifacts, config, adapter_path=artifacts.adapter_v1, adapter_name="v1"
    )
    model.load_adapter(artifacts.adapter_v2, adapter_name="v2")
    model.set_adapter("v1")

    def run(config_name: str) -> None:
        kwargs = {"adapter_names": ["v1"]} if config_name == "per_request_adapter_names" else {}
        _forward_logits(model, inputs, **kwargs)

    for config_name in ("active_adapter", "per_request_adapter_names"):
        for _ in range(config.benchmark_warmups):
            run(config_name)

    schedule = ["active_adapter"] * config.benchmark_trials + ["per_request_adapter_names"] * config.benchmark_trials
    random.Random(config.seed + 3001).shuffle(schedule)
    rows: List[Dict[str, Any]] = []
    for trial, config_name in enumerate(schedule):
        synchronize(device)
        started = time.perf_counter_ns()
        run(config_name)
        elapsed_ms = (time.perf_counter_ns() - started) / 1e6
        rows.append({"trial": trial, "config": config_name, "latency_ms": elapsed_ms})

    frame = pd.DataFrame(rows)
    baseline = frame.loc[frame.config == "active_adapter", "latency_ms"].to_numpy()
    named = frame.loc[frame.config == "per_request_adapter_names", "latency_ms"].to_numpy()
    ratio = float(np.median(named) / np.median(baseline) - 1.0)
    ci_low, ci_high = bootstrap_ratio_ci(named, baseline, seed=config.seed + 3002)
    summary = {
        "active_adapter_median_ms": float(np.median(baseline)),
        "per_request_adapter_names_median_ms": float(np.median(named)),
        "active_adapter_p95_ms": percentile(baseline, 95),
        "per_request_adapter_names_p95_ms": percentile(named, 95),
        "median_overhead_percent": ratio * 100.0,
        "overhead_ci95_percent": [ci_low * 100.0, ci_high * 100.0],
        "trials_per_config": config.benchmark_trials,
    }
    frame.to_csv(output / "named_adapter_overhead_trials.csv", index=False)
    write_json(output / "named_adapter_overhead_summary.json", summary)
    return frame, summary


def _greedy_decode_once(
    model: Any,
    inputs: Mapping[str, torch.Tensor],
    max_new_tokens: int,
    *,
    adapter_names: Optional[List[str]] = None,
) -> Dict[str, float]:
    device = next(model.parameters()).device
    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask", torch.ones_like(input_ids))
    generated = input_ids
    past_key_values = None
    first_token_ms = float("nan")
    started = time.perf_counter_ns()

    with torch.inference_mode():
        for step in range(max_new_tokens):
            prepared = model.prepare_inputs_for_generation(
                generated,
                past_key_values=past_key_values,
                attention_mask=attention_mask,
                use_cache=True,
            )
            if adapter_names is not None:
                prepared["adapter_names"] = adapter_names
            output = model(**prepared)
            synchronize(device)
            if step == 0:
                first_token_ms = (time.perf_counter_ns() - started) / 1e6
            next_token = torch.argmax(output.logits[:, -1, :], dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=-1)
            attention_mask = torch.cat(
                [attention_mask, torch.ones_like(next_token, device=attention_mask.device)], dim=-1
            )
            past_key_values = output.past_key_values

    total_ms = (time.perf_counter_ns() - started) / 1e6
    decode_ms = max(0.0, total_ms - first_token_ms)
    decode_tokens = max(0, max_new_tokens - 1)
    end_to_end_tps = float(max_new_tokens / (total_ms / 1000.0))
    decode_tps = (
        float(decode_tokens / (decode_ms / 1000.0))
        if decode_tokens > 0 and decode_ms > 0
        else float("nan")
    )
    return {
        "ttft_ms": first_token_ms,
        "total_latency_ms": total_ms,
        "generated_tokens": float(max_new_tokens),
        "tokens_per_second": end_to_end_tps,
        "end_to_end_tokens_per_second": end_to_end_tps,
        "decode_tokens_per_second": decode_tps,
    }


def benchmark_autoregressive_generation(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Measure TTFT and decode throughput for active vs request-named adapters."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    model, _, _, inputs = _load_peft_model(
        artifacts, config, adapter_path=artifacts.adapter_v1, adapter_name="v1"
    )
    model.load_adapter(artifacts.adapter_v2, adapter_name="v2")
    model.set_adapter("v1")

    for mode in ("active_adapter", "per_request_adapter_names"):
        for _ in range(max(1, config.benchmark_warmups // 2)):
            names = ["v1"] if mode == "per_request_adapter_names" else None
            _greedy_decode_once(model, inputs, min(3, config.max_new_tokens), adapter_names=names)

    trials = max(5, config.benchmark_trials // 3)
    schedule = ["active_adapter"] * trials + ["per_request_adapter_names"] * trials
    random.Random(config.seed + 3101).shuffle(schedule)
    rows: List[Dict[str, Any]] = []
    for trial, mode in enumerate(schedule):
        names = ["v1"] if mode == "per_request_adapter_names" else None
        metrics = _greedy_decode_once(model, inputs, config.max_new_tokens, adapter_names=names)
        rows.append({"trial": trial, "mode": mode, **metrics})

    frame = pd.DataFrame(rows)
    summary: Dict[str, Any] = {}
    for mode in ("active_adapter", "per_request_adapter_names"):
        subset = frame.loc[frame["mode"] == mode]
        summary[mode] = {
            "median_ttft_ms": float(subset.ttft_ms.median()),
            "p95_ttft_ms": percentile(subset.ttft_ms, 95),
            "median_total_latency_ms": float(subset.total_latency_ms.median()),
            "p95_total_latency_ms": percentile(subset.total_latency_ms, 95),
            "median_end_to_end_tokens_per_second": float(
                subset["end_to_end_tokens_per_second"].median()
            ),
            "median_decode_tokens_per_second": float(
                subset["decode_tokens_per_second"].median()
            ),
        }
    frame.to_csv(output / "generation_trials.csv", index=False)
    write_json(output / "generation_summary.json", summary)
    return frame, summary


def run_natural_hotswap_stress(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """Run concurrent forwards and real PEFT hotswaps without forced pauses.

    This is a stress test, not a proof of safety. Zero observed hybrids only means
    the race did not occur under the tested schedule; the deterministic hook test
    establishes reachability separately.
    """
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    references = _load_references(artifacts)
    model, _, device, inputs = _load_peft_model(artifacts, config)
    start_event = threading.Event()
    result_lock = threading.Lock()
    request_rows: List[Dict[str, Any]] = []
    update_rows: List[Dict[str, Any]] = []

    def request_task(request_id: int) -> None:
        start_event.wait(timeout=30)
        started = time.perf_counter_ns()
        try:
            logits = _forward_logits(model, inputs)
            closest, distances, matched = _classify_logits(logits, references, config.match_atol, config.match_rtol)
            row = {
                "request_id": request_id,
                "latency_ms": (time.perf_counter_ns() - started) / 1e6,
                "observed_result": closest,
                "within_tolerance": matched,
                "committed_version": closest in {"v1", "v2"} and matched,
                "uncommitted_result": not (closest in {"v1", "v2"} and matched),
                "error": None,
                **{f"distance_{name}": value for name, value in distances.items()},
            }
        except BaseException as exc:
            row = {
                "request_id": request_id,
                "latency_ms": (time.perf_counter_ns() - started) / 1e6,
                "observed_result": "error",
                "within_tolerance": False,
                "committed_version": False,
                "uncommitted_result": True,
                "error": repr(exc),
            }
        with result_lock:
            request_rows.append(row)

    def update_task() -> None:
        start_event.wait(timeout=30)
        rng = random.Random(config.seed + 4001)
        for update_id in range(config.stress_updates):
            target = artifacts.adapter_v2 if update_id % 2 == 0 else artifacts.adapter_v1
            version = 2 if update_id % 2 == 0 else 1
            started = time.perf_counter_ns()
            try:
                _hotswap_adapter(model, target, artifacts.adapter_name, device)
                synchronize(device)
                row = {
                    "update_id": update_id,
                    "target_version": version,
                    "latency_ms": (time.perf_counter_ns() - started) / 1e6,
                    "error": None,
                }
            except BaseException as exc:
                row = {
                    "update_id": update_id,
                    "target_version": version,
                    "latency_ms": (time.perf_counter_ns() - started) / 1e6,
                    "error": repr(exc),
                }
            with result_lock:
                update_rows.append(row)
            time.sleep(rng.uniform(0.0, 0.004))

    suite_started = time.perf_counter_ns()
    updater = threading.Thread(target=update_task, name="peft-stress-updater", daemon=True)
    updater.start()
    with concurrent.futures.ThreadPoolExecutor(max_workers=config.stress_workers) as pool:
        futures = [pool.submit(request_task, index) for index in range(config.stress_requests)]
        start_event.set()
        for future in futures:
            future.result(timeout=180)
    updater.join(timeout=180)
    if updater.is_alive():
        raise TimeoutError("stress updater did not finish")
    elapsed_s = (time.perf_counter_ns() - suite_started) / 1e9

    request_frame = pd.DataFrame(request_rows).sort_values("request_id").reset_index(drop=True)
    update_frame = pd.DataFrame(update_rows).sort_values("update_id").reset_index(drop=True)
    summary = {
        "requests": int(len(request_frame)),
        "updates": int(len(update_frame)),
        "request_errors": int(request_frame.error.notna().sum()),
        "update_errors": int(update_frame.error.notna().sum()),
        "uncommitted_results": int(request_frame.uncommitted_result.sum()),
        "observed_v1": int((request_frame.observed_result == "v1").sum()),
        "observed_v2": int((request_frame.observed_result == "v2").sum()),
        "observed_hybrid_A1_B2": int((request_frame.observed_result == "hybrid_A1_B2").sum()),
        "observed_hybrid_A2_B1": int((request_frame.observed_result == "hybrid_A2_B1").sum()),
        "requests_per_second": float(len(request_frame) / elapsed_s),
        "median_request_ms": float(request_frame.latency_ms.median()),
        "p95_request_ms": percentile(request_frame.latency_ms, 95),
        "median_update_ms": float(update_frame.latency_ms.median()),
        "p95_update_ms": percentile(update_frame.latency_ms, 95),
        "interpretation": (
            "This natural stress test estimates observed frequency. It does not prove safety when "
            "no hybrid is observed; use the deterministic real-PEFT race as the reachability proof."
        ),
    }
    request_frame.to_csv(output / "natural_stress_requests.csv", index=False)
    update_frame.to_csv(output / "natural_stress_updates.csv", index=False)
    write_json(output / "natural_stress_summary.json", summary)
    return request_frame, update_frame, summary


def measure_update_and_memory_costs(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Measure hotswap/preload costs and report exact adapter double-buffer bytes."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    metadata = json.loads(Path(artifacts.metadata_path).read_text(encoding="utf-8"))
    adapter_bytes = int(metadata["adapter_tensor_bytes_per_version"])
    rows: List[Dict[str, Any]] = []

    model, _, device, _ = _load_peft_model(artifacts, config)
    synchronize(device)
    before = memory_snapshot(device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    update_times: List[float] = []
    for index in range(max(4, config.benchmark_trials // 3)):
        target = artifacts.adapter_v2 if index % 2 == 0 else artifacts.adapter_v1
        started = time.perf_counter_ns()
        _hotswap_adapter(model, target, artifacts.adapter_name, device)
        synchronize(device)
        update_times.append((time.perf_counter_ns() - started) / 1e6)
    after = memory_snapshot(device)
    rows.append(
        {
            "strategy": "peft_in_place_hotswap",
            "median_update_ms": float(np.median(update_times)),
            "p95_update_ms": percentile(update_times, 95),
            "persistent_rss_delta_mb": (after["rss_bytes"] - before["rss_bytes"]) / (1024 ** 2)
            if min(after["rss_bytes"], before["rss_bytes"]) >= 0
            else float("nan"),
            "persistent_cuda_allocated_delta_mb": (
                after.get("cuda_allocated_bytes", 0) - before.get("cuda_allocated_bytes", 0)
            )
            / (1024 ** 2),
            "peak_cuda_allocated_delta_mb": (
                after.get("cuda_max_allocated_bytes", 0) - before.get("cuda_allocated_bytes", 0)
            )
            / (1024 ** 2),
            "exact_additional_adapter_bytes": 0,
        }
    )
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    named_model, _, device, _ = _load_peft_model(
        artifacts, config, adapter_path=artifacts.adapter_v1, adapter_name="v1"
    )
    synchronize(device)
    before_named = memory_snapshot(device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter_ns()
    named_model.load_adapter(artifacts.adapter_v2, adapter_name="v2")
    synchronize(device)
    load_ms = (time.perf_counter_ns() - started) / 1e6
    after_named = memory_snapshot(device)
    rows.append(
        {
            "strategy": "peft_preload_named_versions",
            "median_update_ms": load_ms,
            "p95_update_ms": load_ms,
            "persistent_rss_delta_mb": (
                after_named["rss_bytes"] - before_named["rss_bytes"]
            )
            / (1024 ** 2)
            if min(after_named["rss_bytes"], before_named["rss_bytes"]) >= 0
            else float("nan"),
            "persistent_cuda_allocated_delta_mb": (
                after_named.get("cuda_allocated_bytes", 0) - before_named.get("cuda_allocated_bytes", 0)
            )
            / (1024 ** 2),
            "peak_cuda_allocated_delta_mb": (
                after_named.get("cuda_max_allocated_bytes", 0)
                - before_named.get("cuda_allocated_bytes", 0)
            )
            / (1024 ** 2),
            "exact_additional_adapter_bytes": adapter_bytes,
        }
    )

    rows.append(
        {
            "strategy": "immutable_adapter_double_buffer_exact_payload",
            "median_update_ms": float("nan"),
            "p95_update_ms": float("nan"),
            "persistent_rss_delta_mb": adapter_bytes / (1024 ** 2),
            "persistent_cuda_allocated_delta_mb": adapter_bytes / (1024 ** 2),
            "peak_cuda_allocated_delta_mb": adapter_bytes / (1024 ** 2),
            "exact_additional_adapter_bytes": adapter_bytes,
        }
    )
    rows.append(
        {
            "strategy": "full_model_mutex",
            "median_update_ms": float("nan"),
            "p95_update_ms": float("nan"),
            "persistent_rss_delta_mb": 0.0,
            "persistent_cuda_allocated_delta_mb": 0.0,
            "peak_cuda_allocated_delta_mb": 0.0,
            "exact_additional_adapter_bytes": 0,
        }
    )

    frame = pd.DataFrame(rows)
    summary = {
        "adapter_tensor_bytes_per_version": adapter_bytes,
        "adapter_tensor_mebibytes_per_version": adapter_bytes / (1024 ** 2),
        "note": (
            "The immutable row reports exact additional LoRA tensor payload. Allocator and runtime "
            "metadata overhead should be measured separately in a production implementation."
        ),
    }
    frame.to_csv(output / "update_memory_costs.csv", index=False)
    write_json(output / "update_memory_costs.json", summary)
    return frame, summary


def _dynamo_stats() -> Dict[str, Any]:
    try:
        counters = torch._dynamo.utils.counters
        return {group: dict(values) for group, values in counters.items()}
    except Exception:
        return {}


def run_torch_compile_hotswap_check(
    artifacts: PEFTArtifacts,
    config: PaperEvalConfig,
    output_dir: os.PathLike[str] | str,
) -> Dict[str, Any]:
    """Check correctness and recompilation indicators around compiled hotswap."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    references = _load_references(artifacts)
    result: Dict[str, Any] = {
        "status": "not_run",
        "backend": config.compile_backend,
    }
    try:
        model, _, device, inputs = _load_peft_model(artifacts, config)
        try:
            from peft.utils.hotswap import prepare_model_for_compiled_hotswap

            prepare_model_for_compiled_hotswap(model, target_rank=config.rank)
        except (ImportError, AttributeError):
            if hasattr(model, "enable_peft_hotswap"):
                model.enable_peft_hotswap(target_rank=config.rank)

        try:
            torch._dynamo.reset()
            torch._dynamo.utils.counters.clear()
        except Exception:
            pass

        compile_started = time.perf_counter_ns()
        compiled_model = torch.compile(model, backend=config.compile_backend, fullgraph=False)
        compile_wrapper_ms = (time.perf_counter_ns() - compile_started) / 1e6

        first_started = time.perf_counter_ns()
        before_logits = _forward_logits(compiled_model, inputs)
        first_call_ms = (time.perf_counter_ns() - first_started) / 1e6
        second_started = time.perf_counter_ns()
        _forward_logits(compiled_model, inputs)
        second_call_ms = (time.perf_counter_ns() - second_started) / 1e6
        stats_before = _dynamo_stats()

        swap_started = time.perf_counter_ns()
        _hotswap_adapter(compiled_model, artifacts.adapter_v2, artifacts.adapter_name, device)
        synchronize(device)
        swap_ms = (time.perf_counter_ns() - swap_started) / 1e6

        post_first_started = time.perf_counter_ns()
        after_logits = _forward_logits(compiled_model, inputs)
        post_first_ms = (time.perf_counter_ns() - post_first_started) / 1e6
        post_second_started = time.perf_counter_ns()
        _forward_logits(compiled_model, inputs)
        post_second_ms = (time.perf_counter_ns() - post_second_started) / 1e6
        stats_after = _dynamo_stats()

        before_class, before_distances, before_match = _classify_logits(
            before_logits, references, config.match_atol, config.match_rtol
        )
        after_class, after_distances, after_match = _classify_logits(
            after_logits, references, config.match_atol, config.match_rtol
        )
        result = {
            "status": "ok",
            "backend": config.compile_backend,
            "compile_wrapper_ms": compile_wrapper_ms,
            "first_call_ms": first_call_ms,
            "second_call_ms": second_call_ms,
            "hotswap_ms": swap_ms,
            "post_swap_first_call_ms": post_first_ms,
            "post_swap_second_call_ms": post_second_ms,
            "before_result": before_class,
            "before_matches_v1": bool(before_class == "v1" and before_match),
            "after_result": after_class,
            "after_matches_v2": bool(after_class == "v2" and after_match),
            "before_distances": before_distances,
            "after_distances": after_distances,
            "dynamo_stats_before_swap": stats_before,
            "dynamo_stats_after_swap": stats_after,
            "interpretation": (
                "Compare unique_graph and recompile counters before and after the swap. Timing alone "
                "is not sufficient evidence of no recompilation."
            ),
        }
    except BaseException as exc:
        result = {
            "status": "error",
            "backend": config.compile_backend,
            "error": repr(exc),
            "guidance": (
                "On CPU or a restricted Colab runtime, set compile_backend='eager' for an API/correctness "
                "smoke test. Use 'inductor' on the target CUDA platform for paper measurements."
            ),
        }
    write_json(output / "torch_compile_hotswap.json", result)
    return result


def _http_json(
    method: str,
    url: str,
    payload: Optional[Mapping[str, Any]] = None,
    timeout: float = 120.0,
) -> Any:
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=data,
        method=method,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        body = response.read().decode("utf-8")
        return json.loads(body) if body.strip().startswith(("{", "[")) else body


def _free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


@contextlib.contextmanager
def launch_vllm_server(
    base_model: str,
    adapter_v1: str,
    adapter_v2: str,
    *,
    port: Optional[int] = None,
    extra_args: Sequence[str] = (),
    log_path: os.PathLike[str] | str = "vllm_server.log",
):

    executable = shutil.which("vllm")
    if executable is None:
        raise RuntimeError("vLLM executable not found; install vllm in a compatible CUDA runtime")
    selected_port = port or _free_port()
    command = [
        executable,
        "serve",
        base_model,
        "--host",
        "127.0.0.1",
        "--port",
        str(selected_port),
        "--enable-lora",
        "--lora-modules",
        f"adapter-v1={adapter_v1}",
        f"adapter-v2={adapter_v2}",
        *list(extra_args),
    ]
    env = os.environ.copy()
    env.setdefault("VLLM_ALLOW_RUNTIME_LORA_UPDATING", "True")
    log_file = open(log_path, "w", encoding="utf-8")
    process = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT, env=env)
    base_url = f"http://127.0.0.1:{selected_port}"
    try:
        deadline = time.time() + 600
        while time.time() < deadline:
            if process.poll() is not None:
                raise RuntimeError(f"vLLM exited early; inspect {log_path}")
            try:
                urllib.request.urlopen(base_url + "/health", timeout=3).read()
                break
            except Exception:
                time.sleep(2)
        else:
            raise TimeoutError("vLLM server did not become healthy")
        yield {"process": process, "base_url": base_url, "command": command}
    finally:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
        log_file.close()


def benchmark_vllm_per_request_lora(
    base_url: str,
    prompts: Sequence[str],
    *,
    max_tokens: int = 16,
    workers: int = 4,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Benchmark vLLM's explicit per-request model/LoRA selection."""
    rows: List[Dict[str, Any]] = []
    lock = threading.Lock()

    def one(index: int, prompt: str) -> None:
        adapter = "adapter-v1" if index % 2 == 0 else "adapter-v2"
        started = time.perf_counter_ns()
        try:
            response = _http_json(
                "POST",
                base_url + "/v1/completions",
                {
                    "model": adapter,
                    "prompt": prompt,
                    "max_tokens": max_tokens,
                    "temperature": 0,
                },
            )
            row = {
                "request_id": index,
                "adapter": adapter,
                "latency_ms": (time.perf_counter_ns() - started) / 1e6,
                "ok": True,
                "response_id": response.get("id") if isinstance(response, dict) else None,
                "error": None,
            }
        except BaseException as exc:
            row = {
                "request_id": index,
                "adapter": adapter,
                "latency_ms": (time.perf_counter_ns() - started) / 1e6,
                "ok": False,
                "response_id": None,
                "error": repr(exc),
            }
        with lock:
            rows.append(row)

    started = time.perf_counter_ns()
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(one, index, prompt) for index, prompt in enumerate(prompts)]
        for future in futures:
            future.result()
    elapsed_s = (time.perf_counter_ns() - started) / 1e9
    frame = pd.DataFrame(rows).sort_values("request_id")
    summary = {
        "requests": len(frame),
        "successful": int(frame.ok.sum()),
        "errors": int((~frame.ok).sum()),
        "median_latency_ms": float(frame.latency_ms.median()),
        "p95_latency_ms": percentile(frame.latency_ms, 95),
        "requests_per_second": float(len(frame) / elapsed_s),
        "semantic_baseline": (
            "Each request names a stable LoRA model. Compare this with immutable runtime pinning, "
            "especially memory residency and update-version behavior."
        ),
    }
    return frame, summary


def recommended_model_matrix() -> pd.DataFrame:
    """Suggested open model configurations for a paper evaluation."""
    rows = [
        {
            "tier": "debug",
            "model_id": "sshleifer/tiny-gpt2",
            "target_modules": "c_attn",
            "ranks": "4",
            "purpose": "fast correctness smoke test only",
        },
        {
            "tier": "small",
            "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
            "target_modules": "q_proj,v_proj",
            "ranks": "4,16,64",
            "purpose": "repeatable modern architecture",
        },
        {
            "tier": "medium",
            "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            "target_modules": "q_proj,v_proj",
            "ranks": "4,16,64",
            "purpose": "1B-class serving evaluation",
        },
        {
            "tier": "large",
            "model_id": "Qwen/Qwen2.5-3B-Instruct",
            "target_modules": "q_proj,v_proj",
            "ranks": "4,16,64",
            "purpose": "multi-GB GPU memory and concurrency evaluation",
        },
        {
            "tier": "optional-7B",
            "model_id": "Qwen/Qwen2.5-7B-Instruct",
            "target_modules": "q_proj,v_proj",
            "ranks": "4,16,64",
            "purpose": "full systems-scale result when hardware permits",
        },
    ]
    return pd.DataFrame(rows)


def run_full_peft_suite(
    output_root: os.PathLike[str] | str,
    config: PaperEvalConfig = PaperEvalConfig(),
    *,
    run_stress: bool = True,
    run_compile: bool = True,
    run_generation: bool = True,
) -> Dict[str, Any]:
    """Run the complete real-PEFT paper evaluation and save raw artifacts."""
    output = Path(output_root)
    if output.exists():
        shutil.rmtree(output)
    output.mkdir(parents=True, exist_ok=True)
    write_json(output / "environment.json", environment_snapshot())
    write_json(output / "config.json", asdict(config))
    recommended_model_matrix().to_csv(output / "recommended_model_matrix.csv", index=False)

    artifacts = build_real_peft_artifacts(output / "peft_artifacts", config)
    race_frame, race_summary = run_real_peft_torn_update(artifacts, config, output / "real_peft_race")
    named_pin_frame, named_pin_summary = run_named_adapter_request_pin_test(
        artifacts, config, output / "named_adapter_pin"
    )
    mutex_frame, mutex_summary = run_full_model_mutex_test(
        artifacts, config, output / "full_model_mutex"
    )
    double_buffer_frame, double_buffer_summary = run_double_buffered_peft_update(
        artifacts, config, output / "double_buffered_peft"
    )
    double_buffer_overhead_frame, double_buffer_overhead_summary = (
        benchmark_double_buffer_router_overhead(
            artifacts, config, output / "double_buffer_router_overhead"
        )
    )
    overhead_frame, overhead_summary = benchmark_named_adapter_overhead(
        artifacts, config, output / "named_adapter_overhead"
    )
    memory_frame, memory_summary = measure_update_and_memory_costs(
        artifacts, config, output / "update_memory"
    )

    generation_frame: Optional[pd.DataFrame] = None
    generation_summary: Optional[Dict[str, Any]] = None
    if run_generation:
        generation_frame, generation_summary = benchmark_autoregressive_generation(
            artifacts, config, output / "generation"
        )

    stress_requests: Optional[pd.DataFrame] = None
    stress_updates: Optional[pd.DataFrame] = None
    stress_summary: Optional[Dict[str, Any]] = None
    if run_stress:
        stress_requests, stress_updates, stress_summary = run_natural_hotswap_stress(
            artifacts, config, output / "natural_stress"
        )

    compile_summary: Optional[Dict[str, Any]] = None
    if run_compile:
        compile_summary = run_torch_compile_hotswap_check(
            artifacts, config, output / "torch_compile"
        )

    safety_comparison = pd.DataFrame(
        [
            {
                "strategy": "PEFT in-place hotswap without coordination",
                "single_request_version_guarantee": False,
                "blocks_inference_during_update": False,
                "keeps_two_versions_resident": False,
                "evidence": race_summary["observed_result"],
            },
            {
                "strategy": "Full-model mutex",
                "single_request_version_guarantee": mutex_summary["safe"],
                "blocks_inference_during_update": True,
                "keeps_two_versions_resident": False,
                "evidence": mutex_summary["request_result"],
            },
            {
                "strategy": "PEFT per-request adapter_names",
                "single_request_version_guarantee": named_pin_summary["request_remained_on_v1"],
                "blocks_inference_during_update": False,
                "keeps_two_versions_resident": True,
                "evidence": named_pin_summary["observed_result"],
            },
            {
                "strategy": "Double-buffered PEFT slots with request leases",
                "single_request_version_guarantee": double_buffer_summary["safe"],
                "blocks_inference_during_update": False,
                "keeps_two_versions_resident": True,
                "evidence": (
                    f"old={double_buffer_summary['in_flight_request_result']}, "
                    f"new={double_buffer_summary['post_publish_request_result']}, "
                    f"recycle_waited={double_buffer_summary['recycle_blocked_while_old_request_live']}"
                ),
            },
            {
                "strategy": "Custom immutable runtime pointer pinning",
                "single_request_version_guarantee": True,
                "blocks_inference_during_update": False,
                "keeps_two_versions_resident": True,
                "evidence": "copy-on-write runtime lifetime; benchmark in original notebook",
            },
        ]
    )
    safety_comparison.to_csv(output / "strategy_safety_comparison.csv", index=False)

    summary = {
        "artifacts": artifacts.as_dict(),
        "real_peft_race": race_summary,
        "named_adapter_pin": named_pin_summary,
        "full_model_mutex": mutex_summary,
        "double_buffered_peft": double_buffer_summary,
        "double_buffer_router_overhead": double_buffer_overhead_summary,
        "named_adapter_overhead": overhead_summary,
        "generation": generation_summary,
        "natural_stress": stress_summary,
        "update_memory": memory_summary,
        "torch_compile": compile_summary,
        "prior_art_note": {
            "peft_issue_804": "https://github.com/huggingface/peft/issues/804",
            "distinction": (
                "Issue #804 describes inter-layer active-adapter switching. The deterministic PEFT "
                "experiment here targets intra-layer A/B tearing during in-place weight replacement."
            ),
        },
        "output_root": str(output),
    }
    write_json(output / "paper_grade_summary.json", summary)

    readme = f"""# Paper-grade PEFT evaluation artifacts

Model: `{config.model_id}`
Targets: `{', '.join(config.target_modules)}`
Rank: `{config.rank}`

PEFT hook experiment establishes that the hybrid execution is reachable.
"""
    (output / "README.md").write_text(readme, encoding="utf-8")

    return {
        "summary": summary,
        "artifacts": artifacts,
        "real_peft_race": race_frame,
        "named_adapter_pin": named_pin_frame,
        "full_model_mutex": mutex_frame,
        "double_buffer_update": double_buffer_frame,
        "double_buffer_overhead": double_buffer_overhead_frame,
        "named_adapter_overhead": overhead_frame,
        "generation_trials": generation_frame,
        "stress_requests": stress_requests,
        "stress_updates": stress_updates,
        "update_memory": memory_frame,
        "safety_comparison": safety_comparison,
    }


Writing paper_grade_peft_eval.py


In [ ]:
import importlib
import inspect
import paper_grade_peft_eval as peft_eval
importlib.invalidate_caches()
importlib.reload(peft_eval)

print("Module reloaded from the corrected source.")
print("Double-buffered router available:", hasattr(peft_eval, "DoubleBufferedPeftRouter"))
print("PaperEvalConfig fields:", [field.name for field in peft_eval.dataclasses.fields(peft_eval.PaperEvalConfig)])


Module reloaded from the corrected source.
Double-buffered router available: True
PaperEvalConfig fields: ['model_id', 'revision', 'trust_remote_code', 'target_modules', 'rank', 'alpha', 'prompt', 'max_length', 'max_new_tokens', 'dtype', 'device', 'seed', 'stress_requests', 'stress_updates', 'stress_workers', 'benchmark_trials', 'benchmark_warmups', 'compile_backend', 'match_atol', 'match_rtol', 'require_reference_separation']


In [ ]:
from importlib.metadata import version, PackageNotFoundError

def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

print({name: installed_version(name) for name in ["torch", "transformers", "peft", "accelerate", "safetensors"]})

# The source now contains the two fixes that previously required runtime patch cells.
source = open("/content/paper_grade_peft_eval.py", encoding="utf-8").read()
assert "torch_device=str(device)" in source
assert 'frame.loc[frame["mode"] == mode]' in source
print("Source-level regression checks passed.")


{'torch': '2.11.0+cu128', 'transformers': '5.13.1', 'peft': '0.19.1', 'accelerate': '1.14.0', 'safetensors': '0.8.0'}
Source-level regression checks passed.


In [ ]:
import importlib
import torch
import paper_grade_peft_eval as peft_eval
importlib.reload(peft_eval)

PAPER_CONFIG = peft_eval.PaperEvalConfig(
    model_id="sshleifer/tiny-gpt2",
    revision=None,  # Mandatory: set an exact commit hash for archival paper runs.
    target_modules=("c_attn",),
    rank=4,
    alpha=8.0,
    dtype="float32",
    stress_requests=40,
    stress_updates=24,
    stress_workers=4,
    benchmark_trials=30,
    benchmark_warmups=5,
    max_new_tokens=12,
    compile_backend="inductor" if torch.cuda.is_available() else "eager",
    match_atol=1e-5,
    match_rtol=1e-4,
    require_reference_separation=True,
)

paper_eval = peft_eval.run_full_peft_suite(
    output_root="/content/secure_atomic_lora_results/paper_grade_peft",
    config=PAPER_CONFIG,
    run_stress=True,
    run_compile=True,
    run_generation=True,
)
paper_eval["summary"]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
W0730 06:31:44.844000 529 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


{'artifacts': {'output_root': '/content/secure_atomic_lora_results/paper_grade_peft/peft_artifacts',
  'model_id': 'sshleifer/tiny-gpt2',
  'adapter_v1': '/content/secure_atomic_lora_results/paper_grade_peft/peft_artifacts/adapter_v1',
  'adapter_v2': '/content/secure_atomic_lora_results/paper_grade_peft/peft_artifacts/adapter_v2',
  'references_path': '/content/secure_atomic_lora_results/paper_grade_peft/peft_artifacts/reference_logits.pt',
  'metadata_path': '/content/secure_atomic_lora_results/paper_grade_peft/peft_artifacts/artifacts_metadata.json',
  'target_layer_name': 'base_model.model.transformer.h.0.attn.c_attn',
  'adapter_name': 'default'},
 'real_peft_race': {'uses_real_peft_layer': True,
  'uses_real_peft_hotswap_api': True,
  'selected_layer': 'base_model.model.transformer.h.0.attn.c_attn',
  'observed_result': 'hybrid_A1_B2',
  'torn_update_reproduced': True,
  'post_swap_result': 'v2',
  'post_swap_matches_v2': True,
  'post_swap_distances': {'v1': 0.019482992589473724

In [ ]:
print("Unsafe real-PEFT deterministic race:")
display(paper_eval["real_peft_race"])

print("Existing explicit per-request PEFT adapter baseline:")
display(paper_eval["named_adapter_pin"])

print("Safe but blocking full-model mutex baseline:")
display(paper_eval["full_model_mutex"])

print("New real-PEFT double-buffered update:")
display(paper_eval["double_buffer_update"])
print(paper_eval["summary"]["double_buffered_peft"])

print("Double-buffer router overhead, isolated over adapter_names:")
display(paper_eval["double_buffer_overhead"].groupby("mode")["latency_ms"].describe())
print(paper_eval["summary"]["double_buffer_router_overhead"])

print("Strategy safety comparison:")
display(paper_eval["safety_comparison"])

print("Update and memory costs:")
display(paper_eval["update_memory"])

if paper_eval["generation_trials"] is not None:
    print("Generation trials:")
    display(paper_eval["generation_trials"].groupby("mode").agg(
        median_ttft_ms=("ttft_ms", "median"),
        p95_ttft_ms=("ttft_ms", lambda x: x.quantile(0.95)),
        median_total_ms=("total_latency_ms", "median"),
        median_end_to_end_tps=("end_to_end_tokens_per_second", "median"),
        median_decode_tps=("decode_tokens_per_second", "median"),
    ))

if paper_eval["stress_requests"] is not None:
    print("Natural concurrent hotswap observations:")
    display(paper_eval["stress_requests"][[
        "request_id", "latency_ms", "observed_result",
        "committed_version", "uncommitted_result", "error"
    ]])


Unsafe real-PEFT deterministic race:


,experiment,selected_layer,observed_result,within_tolerance,is_torn,hotswap_ms,max_abs_error_to_v1,max_abs_error_to_v2,max_abs_error_to_hybrid_A1_B2,max_abs_error_to_hybrid_A2_B1
0,real_peft_hotswap,base_model.model.transformer.h.0.attn.c_attn,hybrid_A1_B2,True,True,19.04116,0.068671,0.076001,0.0,0.031723


Existing explicit per-request PEFT adapter baseline:


,experiment,requested_adapter,global_active_after_switch,observed_result,within_tolerance,single_version,max_abs_error_to_v1,max_abs_error_to_v2,max_abs_error_to_hybrid_A1_B2,max_abs_error_to_hybrid_A2_B1
0,peft_adapter_names_request_pin,v1,v2,v1,True,True,0.0,0.019483,0.068671,0.036948


Safe but blocking full-model mutex baseline:


...

,strategy,median_update_ms,p95_update_ms,persistent_rss_delta_mb,persistent_cuda_allocated_delta_mb,peak_cuda_allocated_delta_mb,exact_additional_adapter_bytes
0,peft_in_place_hotswap,1.691467,2.928586,0.011719,0.000000,0.001953,0
1,peft_preload_named_versions,14.578936,14.578936,0.011719,0.001953,0.003906,256
2,immutable_adapter_double_buffer_exact_payload,NaN,NaN,0.000244,0.000244,0.000244,256
3,full_model_mutex,NaN,NaN,0.000000,0.000000,0.000000,0


Generation trials:


,median_ttft_ms,p95_ttft_ms,median_total_ms,median_end_to_end_tps,median_decode_tps
mode,,,,,
active_adapter,3.802111,8.373104,56.933844,210.820676,207.796280
per_request_adapter_names,4.586708,7.454062,66.877248,179.524791,178.134566


Natural concurrent hotswap observations:


,request_id,latency_ms,observed_result,committed_version,uncommitted_result,error
0,0,112.315336,v2,True,False,None
1,1,63.038540,v1,True,False,None
2,2,69.260974,v1,True,False,None
3,3,81.522041,v1,True,False,None
4,4,73.716533,v1,True,False,None
5,5,84.971411,v1,True,False,None
6,6,58.082610,v1,True,False,None
7,7,69.029433,v2,True,False,None
8,8,59.254330,v2,True,False,None
9,9,78.001970,v2,True,False,None


In [ ]:
display(peft_eval.recommended_model_matrix())

RUN_SCALE_MATRIX = False
SCALE_RUNS = [
    {"model_id": "Qwen/Qwen2.5-0.5B-Instruct", "target_modules": ("q_proj", "v_proj"), "rank": 4},
    {"model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "target_modules": ("q_proj", "v_proj"), "rank": 16},
    {"model_id": "Qwen/Qwen2.5-3B-Instruct", "target_modules": ("q_proj", "v_proj"), "rank": 64},
]

if RUN_SCALE_MATRIX:
    from dataclasses import replace
    scale_results = {}
    for spec in SCALE_RUNS:
        run_config = replace(
            PAPER_CONFIG,
            model_id=spec["model_id"],
            target_modules=spec["target_modules"],
            rank=spec["rank"],
            revision=None,  # Replace with an exact commit hash.
            dtype="float16" if torch.cuda.is_available() else "float32",
            match_atol=1e-3 if torch.cuda.is_available() else 1e-5,
        )
        slug = run_config.model_id.replace("/", "__") + f"__r{run_config.rank}"
        scale_results[slug] = peft_eval.run_full_peft_suite(
            f"/content/secure_atomic_lora_results/model_matrix/{slug}",
            run_config,
            run_stress=True,
            run_compile=True,
            run_generation=True,
        )


,tier,model_id,target_modules,ranks,purpose
0,debug,sshleifer/tiny-gpt2,c_attn,4,fast correctness smoke test only
1,small,Qwen/Qwen2.5-0.5B-Instruct,"q_proj,v_proj","4,16,64",repeatable modern architecture
2,medium,TinyLlama/TinyLlama-1.1B-Chat-v1.0,"q_proj,v_proj","4,16,64",1B-class serving evaluation
3,large,Qwen/Qwen2.5-3B-Instruct,"q_proj,v_proj","4,16,64",multi-GB GPU memory and concurrency evaluation
4,optional-7B,Qwen/Qwen2.5-7B-Instruct,"q_proj,v_proj","4,16,64",full systems-scale result when hardware permits


In [ ]:

RUN_VLLM_BASELINE = False

if RUN_VLLM_BASELINE:
    artifacts = paper_eval["artifacts"]
    if PAPER_CONFIG.model_id == "sshleifer/tiny-gpt2":
        raise ValueError("Use a vLLM-supported modern model and rebuild PAPER_CONFIG first.")
    prompts = [PAPER_CONFIG.prompt + f" Request {i}." for i in range(32)]
    log_path = "/content/secure_atomic_lora_results/vllm_server.log"
    with peft_eval.launch_vllm_server(
        PAPER_CONFIG.model_id,
        artifacts.adapter_v1,
        artifacts.adapter_v2,
        extra_args=("--max-loras", "2", "--max-lora-rank", str(PAPER_CONFIG.rank)),
        log_path=log_path,
    ) as server:
        vllm_trials, vllm_summary = peft_eval.benchmark_vllm_per_request_lora(
            server["base_url"], prompts, max_tokens=PAPER_CONFIG.max_new_tokens, workers=4
        )
    vllm_trials.to_csv(
        "/content/secure_atomic_lora_results/vllm_per_request_trials.csv", index=False
    )
    peft_eval.write_json(
        "/content/secure_atomic_lora_results/vllm_per_request_summary.json", vllm_summary
    )
    display(vllm_trials)
    print(vllm_summary)


In [ ]:

import shutil
archive = shutil.make_archive(
    "/content/secure_atomic_lora_results",
    "zip",
    "/content/secure_atomic_lora_results",
)
print(archive)


/content/secure_atomic_lora_results.zip
